# Circuit Tracing in VLMs — reduced reproduction

**Paper:** Yang, Xiong, Qian, Nahrstedt, Wu. *Circuit Tracing in Vision–Language Models: Understanding the Internal Mechanisms of Multimodal Thinking.* CVPR Findings 2026, pp. 3322–3331.

## Feasibility: DOES NOT FIT — this is a reduced run

The paper trained transcoders with "a batch size of 12 and runs for 30,000 steps on 8 H100 GPUs for approximately 60 hours" (Sec. 4.1), one transcoder per MLP for all 34 decoder layers, and computed attribution graphs at 20 min/graph on an H100 in bfloat16 (Sec. 4.2). Two T4s (16 GB, sm_75, no bf16, 12 h session) cannot hold 34 transcoders, cannot run 30,000 steps, and cannot use bfloat16.

**The paper reports no accuracy metric of any kind.** It has no GQA experiment, no exact-match scoring, and no grounding evaluation. Its quantitative results are FVU (Eq. 3) and dead-latent percentage (Fig. 4). The GQA exact-match and box-IoU scoring below are an **adaptation**, labelled as such in the `fidelity` column.

## Deviation table

| Paper's value | This notebook | Effect on results |
|---|---|---|
| 8×H100 80 GB, ~60 h | 2×T4 16 GB, <=12 h session | Sets every reduction below |
| Transcoders on all 34 MLP layers | `TRANSCODER_LAYERS = [3, 15, 27]` (the layers plotted in Fig. 4) | Replacement model is *partial*: 31 of 34 MLPs stay intact. Replacement accuracy closer to base than the paper's full replacement; attribution graphs contain feature nodes at 3 layers only |
| N_latents = 64 (grid {32,64,128}) | N_latents = 32 (smallest in-grid value) | Fig. 4 top shows Dead PCT depends strongly on N_latents; FVU will differ. 419 M params/transcoder vs 839 M |
| 30,000 steps, batch 12 | **`TRAIN_STEPS = 50`**, wall-clock cap `MAX_SECONDS_PER_RUN = 2400`; actual completed steps measured and written to `Training Steps` | Measured cost is ~29 s/step on 2×T4 — the fp32 transcoder matmul dominates, not the Gemma forward — so 3,000 steps would be ~24 h for one of six runs. 50 steps is 0.17% of the paper's step budget and ~1.2 M tokens against the paper's ~740 M. FVU will be far from converged; treat it as an early-training reading, not the paper's FVU |
| Dead PCT over a 30,000-step run (Fig. 4) | Measured over the final 10% of a 50-step run = 5 steps | **Not comparable to Fig. 4.** With k=48 and 5 steps × 12 sequences × 2048 tokens, at most a few million latent slots can fire against 81,920 latents, and the transcoder has barely left initialisation. Cell 5 prints this caveat next to the number and the results table repeats it in `notes` |
| Linear warmup over 1,000 of 30,000 steps (3.33%) | Same 3.33% ratio of the *actual* step budget | Keeping 1,000 literal would make warmup ~50% of the run and distort the LR curve |
| ImageNet 144,000 / Cauldron 72,000 / text 144,000 | Same 2:2:1 mixture ratio, streamed, truncated to whatever the step cap consumes | Less concept coverage; features less monosemantic |
| SmolLM2 pretraining mixture (Fig. 2) | `smollm-corpus / fineweb-edu-dedup` only (A15) | Narrower text domain; affects the text-only FVU arm of Fig. 4 |
| `google/gemma-3-4b-it` (Sec. 4 "Model") | `unsloth/gemma-3-4b-it` (ungated mirror) | Expected zero numerical deviation — same weight lineage per the HF model tree. **Unverified by hash**; Cell 1b asserts the architecture matches Sec. 4 and stops on mismatch |
| `ILSVRC/imagenet-1k` at native resolution (Table 1) | `evanarlian/imagenet_1k_resized_256` (ungated; short side resized to 256) | **Real deviation.** Gemma-3 resizes to 896×896 (Sec. 4), so a 256-short-side JPEG is upsampled ~3.5×. Fine detail that early-layer features key on ("down to digits or textures", Sec. 5) is absent from the source. Affects the multimodal arm only |
| bfloat16 for attribution (Sec. 4.2); training precision NOT SPECIFIED | **fp32**, model sharded lopsidedly across both T4s (`max_memory={0:'12GiB', 1:'5GiB'}`) | T4 has no bf16, and fp16 is not a usable substitute: measured on this setup, layer-3 MLP activations are finite in fp16 but the layer-15 MLP output overflows (fp16 max 65504) and the captured target contains inf before any gradient step. fp32 is numerically closer to the paper's bf16 than fp16 is — same exponent range, more mantissa — so this costs speed, not fidelity. fp32 Gemma is 16.02 GiB and needs both cards; **this configuration requires 2 GPUs and raises on 1** |
| Attention kernel NOT SPECIFIED | sdpa for the training capture forward, eager for attribution | At batch 12 × 2048, eager materialises a 12×8×2048×2048 fp32 attention matrix (1.50 GiB, 3.00 GiB with the softmax upcast) that does not fit beside fp32 weights. Attribution needs eager because freezing the softmax (Sec. 3.2) works by patching `F.softmax`, which fused kernels never call; at ~289 prompt tokens eager is cheap. Cell 7 counts patch invocations and raises if it was a silent no-op |
| Attribution: m = 7500 features, 20 min/graph on H100 | Thresholds 0.8/0.98 and <=10 logit nodes kept; target set capped at 128 features per trained layer, on 3 prompts | Graph is a subgraph of the paper's; edge weights computed exactly per Eqs. 5–7 |
| Attention maps precomputed for 28,000 images (~2 TB) | Computed on the fly for the eval subset, nothing cached | Identical math, no storage |
| — (paper has no VQA eval) | GQA CoT val, `EVAL_N = 500` of 9,855 records | **Not the paper's score.** A 500-record subset score is not a 9,855-record score. Lowered from 1000 to protect the 12 h session, since training now costs up to ~6 h and fp32 greedy generation runs twice |
| — | Grounding IoU eval, `GROUND_N = 500` | Adaptation; no paper counterpart |

## Paper errata

Discrepancies where the paper's stated value is wrong, as distinct from assumptions (where the paper is silent). Recorded so the deviation is attributable to the paper, not to this reproduction.

| Paper says | Actual | Evidence |
|---|---|---|
| "34 attention heads" (Sec. 4, "Architecture") | 8 query heads, 4 KV heads, head_dim 256 | `Gemma3TextConfig` on the loaded checkpoint. 2560/34 is not an integer, so the stated figure cannot describe any model; Gemma 3 also decouples head_dim from `hidden_size // num_heads`, so the two are unrelated in the first place. The value appears to be the layer count (34, stated in the same sentence) copied into the heads slot. Nothing in this notebook is sized from the head count, so Cell 1b reports the discrepancy and continues |

Every other architectural figure in Sec. 4 — 34 layers, d_model 2560, d_ff 10240, SigLIP patch 14 at 896×896, 256 image tokens — matches the checkpoint and is hard-asserted in Cell 1b.

| Cauldron sampled evenly from all 50 subsets (Sec. 4.1) | Only subsets that pass a streamability probe (48/50 observed; `clevr_math` and `okvqa` excluded), visited round-robin with one stream open at a time, 4 records per visit | Some subsets (CLEVR observed) store Image features as absolute paths on the dataset author's filesystem (`/fsx/m4/...`) instead of embedded bytes, so decoding raises `FileNotFoundError` and they cannot stream at all. A defect in the dataset, not the notebook. Coverage is narrower than "evenly from 50 subsets"; individual unreadable records mid-stream are counted and reported, never substituted. Holding all 48 iterators open with 100-image shuffle buffers exhausted host RAM (~30 GiB) and killed the kernel, so exactly one stream is open at a time. A 50-step run draws ~125 Cauldron records against 192 available per full pass, so no subset is revisited; on a longer run a revisited subset would reopen from the start and repeat its first records |
| Full 34-layer forward | Decoder truncated to layers 0..L when capturing layer L | **Exact, not an approximation** — layer L's MLP input depends only on layers below it. Measured cost was ~63 s/step for every target layer because the full forward ran regardless; layer 3 needs 4 of 34 layers. Costs one model reload per target layer plus one full reload for Cells 6–7 |

## Assumptions (paper is silent on each of these)

| # | Assumption | Why needed |
|---|---|---|
| A1 | `d_feat = N_latents · d_model` **per layer**; the "·34" in Sec. 4.1 read as the total across 34 layers | Literal reading gives 5.57 M features and 28.5 B params *per layer* (970 B total), inconsistent with 8×H100/60 h |
| A2 | Loss = MSE between `TC(x)` and `MLP(x)` | Paper says "reconstruction error"; Eq. 3 uses MSE |
| A3 | Init: `W_enc` Kaiming-uniform, `W_dec = W_enc^T`, `b_enc = 0`, **`b_dec` seeded from the mean of `MLP(x)`**, and **`W_dec` rescaled so the initial output perturbation is 0.1 × the target's std** | Init NOT SPECIFIED, but Sec. 4.1 states the framework was built "on top of Sparsify", whose convention seeds the decoder bias from the data mean. Measured with `b_dec = 0`: error-node RMS equalled `sqrt(MSE)` to four digits on all three layers, i.e. `TC(x)` contributed nothing, and FVU sat at `1 + mean²/Var` (1.26 / 2.08 / 1.11). A zero bias cannot emit even the constant mean of the target. The rescale addresses a second scale problem: with Kaiming init the decoder's initial magnitude is set by the weights (std ~0.028) regardless of the target, whose std ranges ~74× across layers (3.77 at layer 3, 0.051 at layer 15). Measured after the bias fix alone: layers 3/3-mm reached FVU 0.519/0.575 but layers 15/15-mm sat at 1.698/1.333, worse than the mean predictor. The rescale makes the starting FVU ~1.01 at every layer. `W_dec` stays non-zero so the encoder still receives gradient |
| A4 | AdamW defaults beta=(0.9,0.999), eps=1e-8, weight_decay=0.0 | NOT SPECIFIED |
| A5 | Constant LR after linear warmup | Post-warmup schedule NOT SPECIFIED |
| A6 | Dead latent = never enters TopK during the final 10% of the run | Threshold/window NOT SPECIFIED |
| A7 | Rollout: K = 4 last vision layers, q = 0.25 lowest-entropy heads, pooling block b = 4 (64×64 -> 16×16 = 256) | K and q NOT SPECIFIED; b inferred from the 4096->256 pooling in Sec. 4 |
| A8 | Rollout saliency map = mean over rows of `R_vis` | Paper interprets each row separately; a single map needs an aggregation |
| A9 | Edge pruning eps = 1e-4 × max abs(A) on the graph | eps NOT SPECIFIED |
| A10 | GQA prompt = Gemma-3 chat template + "Answer with a single word or short phrase."; greedy, max 16 new tokens | No VQA protocol in the paper |
| A11 | Answer normalisation: lowercase; delete apostrophes; replace every other punctuation mark with a space; drop a leading article unless it is the only token; collapse whitespace. Exact string match on the result | Not in paper. Hyphens become spaces rather than being deleted so a hyphenated prediction scores equal to unhyphenated gold — deleting them would normalise "hot-dog" to "hotdog" and miss against gold "hot dog". Cell 3 runs a printed self-test over eight cases |
| A12 | Grounding GT = union of `bboxs`; prediction = bbox of pixels >= 0.5·max of rollout map; hit at IoU >= 0.5 | Not in paper. **Measured behaviour:** the rollout map is extremely peaked, so the predicted box averages ~1 cell of the 16×16 pooled map (0.005 of the image) against a ground-truth region of 0.174, putting IoU ≥ 0.5 out of reach by construction — the area-ratio ceiling is ~0.22. Cell 7 prints the full distribution. This is a property of the attention maps, which Sec. 7 itself reports "sometimes fail to localize relevant regions", not a scoring bug. `MAP_THRESH_FRAC`, `ROLLOUT_K`, `ROLLOUT_Q` and A8 have **not** been tuned to raise the score |
| A13 | Full-frame record = union box area >= 0.75 of image area | "covers most of the frame" is unquantified |
| A14 | ImageNet caption template `"This is a photo of a {label}."` | Table 1 says "1 image; Caption"; ImageNet ships labels, not captions |
| A15 | SmolLM2 text proxy = `HuggingFaceTB/smollm-corpus`, config `fineweb-edu-dedup` | Mixture weights of the SmolLM2 pretraining mix are not given |
| A16 | Non-transcoded MLP layers contribute detached constants in the attribution graph | Consequence of the reduced layer set; paper transcodes all 34 |
| A17 | Image processor pinned to `use_fast=False`, the variant the checkpoint was saved with | The transformers default for `use_fast` is version-dependent and the two variants give slightly different pixel values; pinning stops preprocessing drifting across versions |

## Resolved: the prompt carried a doubled BOS

The first end-to-end run scored **0.0 on both the base and the transcoder-replaced
model**, emitting proper nouns (`Richard`, `Park`) rather than GQA answers. Because both
models scored 0, it was not the transcoders. The Cell 3 check confirmed the cause
directly:

```
BOS token id 2 appears 2x in the prompt
prompt head: '<bos><bos><start_of_turn>user\n\n\n<start_of_image>...'
```

`apply_chat_template(tokenize=False)` already emits `<bos>`, and the processor's
tokenizer prepended a second. Gemma stops behaving like an instruction-tuned chat model
on a doubled BOS and continues text instead. The rest of the prompt was correct — 256
image tokens, the question, the instruction, and the `<start_of_turn>model` marker.

**Fix:** `encode_vqa` and the multimodal branch of `encode_training_batch` now pass
`add_special_tokens=False`, which `Gemma3Processor` forwards to the tokenizer as a
recognised `TextKwargs` field. The text-only training branch feeds *raw* text and keeps
the default, correctly prepending a single `<bos>`. Cell 3 now requires **exactly one**
BOS and raises on 0 or 2+.

Accuracy figures produced before this fix are invalid. The first run after it is the
first whose `Acc` column means anything.

**What exact match looks like after the fix.** Predictions are now real answers, and the
misses are mostly *near* misses against GQA's single-word gold: `Clock tower` vs `clock`,
`Glass vase` vs `glass`, alongside clean hits like `Suitcase` vs `suitcase`. Normalised
exact match (A11) counts those near misses as wrong. That is the metric behaving as
defined, not a bug — the paper specifies no VQA metric at all, so exact match is this
notebook's adaptation and is labelled as such in `fidelity`. A containment-style score
(gold appearing inside the prediction) would read higher and could be added as an extra
column, but has deliberately **not** been added: swapping to a laxer metric after seeing
the numbers is fitting the metric to the result.

## No results are pre-filled

Every number in `results.csv` is computed at runtime from real data. Nothing in this notebook fabricates, samples, or defaults a metric. `ROC-AUC` and `PR-AUC` are `N/A` because the task emits short answer strings, not class probabilities — building an AUC from correctness flags returns 1.0 by construction and measures nothing.

**Licences still apply.** The ungated mirrors redistribute the artifacts, not the terms: the Gemma Terms of Use govern the weights, and the ImageNet terms (non-commercial research and educational use only) govern the images.

In [ ]:
# ============================================================================
# CELL 0 — PINNED INSTALLS
# Only pure-Python packages are pinned. numpy / pandas / pillow / torch are taken
# from the Kaggle image UNCHANGED and are pinned to their existing versions via a
# constraints file, because pip-replacing a compiled package inside a live kernel
# leaves mixed Python/.so files and raises
#     ValueError: numpy.dtype size changed ... Expected 96 ... got 88
# on the next import. That is only recoverable by restarting the kernel, which
# papermill cannot do mid-notebook.
# ============================================================================
import subprocess, sys, importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

# Compiled stack that must not move. Recorded, then frozen via -c constraints.
COMPILED = ("numpy", "pandas", "pillow", "torch", "torchvision")
PRE = {p: ver(p) for p in COMPILED}
print("container versions BEFORE install:")
for p, v in PRE.items():
    print(f"  {p:12s} {v}")

CONSTRAINTS = "/tmp/constraints.txt"
with open(CONSTRAINTS, "w") as f:
    for p, v in PRE.items():
        if v:
            f.write(f"{p}=={v}\n")

# Pure-Python pins. transformers is pinned because CELL 7 reaches into
# transformers.models.gemma3.modeling_gemma3 internals (Gemma3RMSNorm.forward).
PINS = [
    "transformers==4.53.2",
    "accelerate==1.8.1",
    "datasets==3.6.0",
    "huggingface_hub==0.33.0",
]
cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts",
       "-c", CONSTRAINTS, *PINS]
print("\nrunning:", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
sys.stdout.write(r.stdout[-4000:])
sys.stderr.write(r.stderr[-4000:])
if r.returncode != 0:
    raise RuntimeError(
        f"pip install failed with exit code {r.returncode}. See output above. "
        "If the conflict is between a pin and the frozen compiled stack, change "
        "the pin — do not relax the constraints file."
    )

POST = {p: ver(p) for p in COMPILED}
moved = {p: (PRE[p], POST[p]) for p in COMPILED if PRE[p] != POST[p]}
if moved:
    raise RuntimeError(
        f"pip moved a compiled package despite the constraints file: {moved}. "
        "The kernel now has a mixed ABI. Restart the kernel and rerun; do not continue."
    )

# Smoke test: exercise the exact numpy paths that the ABI break used to kill.
import numpy as _np, pandas as _pd
_np.random.seed(0)
assert _np.zeros(3, dtype=_np.float32).sum() == 0.0
assert len(_pd.DataFrame({"a": [1, 2]})) == 2
print("\nnumpy/pandas ABI smoke test passed")

print("\nresolved environment:")
for p in COMPILED:
    print(f"  {p:16s} {ver(p)}")
for p in PINS:
    name = p.split("==")[0]
    print(f"  {name:16s} {ver(name)}")
print(f"  {'python':16s} {sys.version.split()[0]}")
print("\ninstalls done")

In [ ]:
# ============================================================================
# CELL 1 — CONFIG  +  ENVIRONMENT AND DATA INTEGRITY CHECKS
# Stops hard if anything is wrong. No substitute images are ever generated.
# ============================================================================
import os
# Must be set before torch initialises CUDA. The 12 GiB logits allocation that
# used to OOM here left large reserved-but-unallocated blocks; expandable segments
# stop that fragmentation from compounding across the six training runs.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# torch.compile is disabled for the whole notebook, set before torch is imported.
# It has cost this reproduction twice:
#   1. A compiled graph never calls the Python F.softmax that CELL 7 patches to freeze
#      attention (Sec. 3.2), so the linearisation would be silently wrong.
#   2. inductor cannot lower the in-place latent edit the steering hook performs
#      (Eq. 9/10) and raised
#        BackendCompilerFailed: IndexError ... While executing setitem(z_sparse, ...)
# Nothing here benefits from compilation: every measured cost is matmul-bound.
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

import sys, json, math, time, random, gc, warnings
import numpy as np
import torch
from PIL import Image

# ----------------------------- CONFIG ---------------------------------------
class CFG:
    # ---- paths (both mount points) ----
    ANN_PATH   = "/kaggle/input/notebooks/khoangoo/test-dataset-visual-cot/visual-cot/cot_with_detailed_reasoning_steps/gqa_cot_val.jsonl"
    IMAGE_ROOT = "/kaggle/input/datasets/lyte69/gqa-images/images"
    OUT_DIR    = "/kaggle/working"

    EXPECTED_RECORDS       = 9855
    EXPECTED_UNIQUE_IMAGES = 5422

    # ---- model (Sec. 4 "Model"/"Architecture") ----
    # DEVIATION: ungated mirror of google/gemma-3-4b-it. Same weight lineage per the
    # HF model tree; architecture is asserted against Sec. 4 in Cell 1b below.
    MODEL_ID   = "unsloth/gemma-3-4b-it"
    D_MODEL    = 2560     # Sec. 4 "Architecture"
    N_LAYERS   = 34       # Sec. 4 "Architecture"
    D_FF       = 10240    # Sec. 4 "Architecture"
    HF_TOKEN   = os.environ.get("HF_TOKEN", None)   # optional; no repo below is gated

    # T4 (sm_75) has no bfloat16; the paper runs attribution in bfloat16 (Sec. 4.2).
    # fp16 is NOT a usable substitute here: measured on this setup, the layer-3 MLP
    # activations are finite but the layer-15 MLP output overflows fp16 (max 65504)
    # and the captured tensor contains inf before any gradient step. Gemma-3's residual
    # stream grows with depth, so shallow layers survive fp16 and mid-stack ones do not.
    # fp32 is also numerically closer to the paper's bf16 than fp16 is (same exponent
    # range, more mantissa), so this costs speed, not fidelity.
    DTYPE = "float32"     # "float32" | "float16" (float16 will fail the CELL 5 probe)

    # fp32 Gemma is 16.02 GiB and does not fit one 14.56 GiB T4, so it is sharded.
    # A 12+5=17 GiB budget nominally exceeds 16.02 but accelerate still spilled weights
    # to DISK ("Some parameters are on the meta device"), so the caps need real slack,
    # not a hair's breadth. With 11+13 accelerate fills cuda:0 to 11 GiB and puts the
    # remaining ~5 GiB on cuda:1, leaving 3.56 GiB free on cuda:0 for activations and
    # 9.54 GiB on cuda:1 for the 6.25 GiB transcoder. CELL 5 audits actual placement
    # and raises if anything landed on meta/cpu -- a disk-offloaded run is not a result.
    # Caps are only honoured with device_map="sequential" (see load_gemma). Under
    # device_map="auto" accelerate rewrites them via get_balanced_memory and splits
    # evenly regardless.
    #
    # Computed at load time, not fixed: the truncated model size varies with the target
    # layer (5.47 / 9.69 / 13.91 GiB for layers 3 / 15 / 27). A fixed 12 GiB cuda:0 cap
    # is fine for layers 3 and 15 -- both fit entirely on cuda:0 -- but spills 2.46 GiB
    # onto cuda:1 for layer 27, leaving 3.60 GiB where the transcoder needs 7.69.
    # Reserving a fixed activation headroom on cuda:0 instead pushes as much of the
    # model there as will fit, whatever its size.
    MAX_MEMORY          = None    # None -> computed from the reserves below
    GPU0_RESERVE_GIB    = 1.6     # forward activations (MLP transient at mb=2 is 0.47)
    GPU1_RESERVE_GIB    = 0.4

    # Decoder init scale. b_dec fixed the mean, but the decoder's initial output
    # magnitude is set by the WEIGHTS, not the target: Kaiming on (81920, 2560) gives
    # std ~0.028 either way, while target std ranges from 3.77 (layer 3) to 0.051
    # (layer 15). Measured: layers 3/3mm reached FVU 0.519/0.575 while layers 15/15mm
    # sat at 1.698/1.333 -- worse than predicting the mean, because at layer 15 the init
    # perturbation swamps the signal. Rescaling W_dec so the initial perturbation is
    # this fraction of the target's std makes the starting FVU ~1+alpha^2 at every
    # layer, independent of scale. See ASSUMPTION A3.
    DECODER_INIT_SCALE  = 0.1

    # Micro-batch for the capture forward ONLY. Mathematically neutral: the transformer
    # has no cross-sample interaction, so splitting a batch of 12 into 4 forwards of 3
    # yields bit-comparable activations, and all 24576 tokens still form ONE optimiser
    # step at the paper's batch size of 12 (Sec. 4.1). Purely a memory measure: the
    # Gemma MLP holds three 12x2048x10240 fp32 intermediates at once inside
    # down_proj(act(gate(x)) * up(x)) = 2.81 GiB, which does not fit beside fp32
    # weights. At micro-batch 3 that transient drops to 0.70 GiB.
    # 2, not 3: cuda:0 now carries ~13 GiB of weights and has ~1.5 GiB free, so the
    # forward transient must stay small. 0.47 GiB at mb=2 vs 0.70 GiB at mb=3.
    MICRO_BATCH = 2

    # sdpa for the training capture forward: at batch 12 x 2048 tokens, eager would
    # materialise a 12x8x2048x2048 fp32 attention matrix (1.50 GiB, 3.00 GiB with the
    # softmax upcast) that does not fit alongside fp32 weights. CELL 7 switches to eager
    # for attribution only, where the sequence is ~289 tokens -- the softmax patch that
    # freezes nonlinearities (Sec. 3.2) requires eager, and CELL 7 verifies it fired.
    ATTN_TRAIN = "sdpa"
    ATTN_ATTRIBUTION = "eager"

    # ---- transcoder hyperparameters (Sec. 3.1, Sec. 4.1) ----
    K_TOPK          = 48        # Sec. 4.1: "for our experiment we set k to 48"
    N_LATENTS       = 32        # Sec. 4.1 grid {32,64,128}; DEVIATION from default 64
    LR_BASE         = 2e-4      # Sec. 4.1
    LR_REF_NUMER    = 2 ** 14   # Sec. 4.1: 2^14
    BATCH_SIZE      = 12        # Sec. 4.1
    PAPER_STEPS     = 30000     # Sec. 4.1
    PAPER_WARMUP    = 1000      # Sec. 4.1
    TEXT_SEQ_LEN    = 2048      # Table 1: "2048 text tokens"

    # ---- reduced-run controls (see deviation table) ----
    TRANSCODER_LAYERS     = [3, 15, 27]   # layers plotted in Fig. 4
    TRAIN_MODES           = ["text", "multimodal"]  # Fig. 4 bottom ablation
    # DEVIATION from the paper's 30,000. Measured cost on 2xT4 is ~29 s/step
    # (the fp32 transcoder matmul dominates, not the Gemma forward), so 3,000
    # steps would be ~24 h for ONE of the six runs. 50 steps x 6 runs is ~2.5 h.
    TRAIN_STEPS           = 50            # DEVIATION from 30000
    # Decoder-bias init. Sec. 4.1 says the framework was built "on top of Sparsify",
    # whose convention seeds the decoder bias from the data mean. Measured with b_dec=0,
    # all three layers gave error-node RMS == sqrt(MSE) to four digits, i.e. TC(x)
    # contributed nothing, and FVU sat at 1 + mean^2/Var (1.26 / 2.08 / 1.11). A zero
    # bias cannot emit even the constant mean of MLP(x), so the run must climb back to
    # FVU 1.0 before explaining any variance. See ASSUMPTION A3.
    BDEC_INIT_BATCHES     = 2

    # Keep only decoder layers 0..L when capturing layer L. Layers above the capture
    # point cannot influence it, so this is exact, not an approximation. Measured cost
    # was ~63 s/step regardless of target layer because the full 34-layer forward ran
    # every time; layer 3 needs 4 of 34 layers.
    TRUNCATE_DECODER      = True

    # Some the_cauldron subsets (e.g. CLEVR) store Image features as paths on the
    # dataset author's filesystem (/fsx/m4/...) rather than embedded bytes, so they
    # cannot stream at all. Probe once, keep what works, record what was dropped.
    CAULDRON_MAX_FAILS    = 3
    # Hold ONE Cauldron stream open at a time. Holding all 48 with shuffle(buffer_size=100)
    # buffers 4,800 decoded PIL images plus 48 parquet row groups and killed the kernel on
    # host RAM (~30 GiB on Kaggle). MIX_RATIO gives Cauldron 1/5 of 12 samples per step, so
    # a 50-step run needs only ~125 Cauldron examples; 48 subsets x 4 per visit = 192 per
    # full pass covers that without a second visit.
    CAULDRON_TAKE_PER_VISIT   = 4
    CAULDRON_SHUFFLE_BUFFER   = 16     # cheap now that only one stream is open
    IMAGENET_SHUFFLE_BUFFER   = 64
    HOST_RAM_ABORT_FRAC       = 0.88   # raise before the kernel is killed silently
    # Raised from 2400: at 2400 every text run was truncated at 38-39 of 50 steps, so
    # the tail window for Dead PCT never opened and it came out NaN. With truncation the
    # deepest run (layer 27, 28 of 34 layers) is the binding case at ~58 s/step.
    MAX_SECONDS_PER_RUN   = 3600          # hard wall-clock cap per run
    WARMUP_FRAC           = PAPER_WARMUP / PAPER_STEPS   # 3.33 %, ratio preserved
    # Memory only, mathematically neutral: each chunk's loss is scaled by its share
    # of the batch and backward()-accumulated, with one opt.step() per batch.
    # At d_feat=81920 one chunk holds several 512x81920 fp32 tensors through backward.
    TOKEN_CHUNK           = 512           # tokens per transcoder fwd/bwd chunk
    FVU_EVAL_BATCHES      = 8             # held-out batches for final FVU
    DEAD_TAIL_FRAC        = 0.10          # A6

    # ---- dataset ids for transcoder training (Table 1) ----
    TEXT_REPO      = "HuggingFaceTB/smollm-corpus"     # A15
    TEXT_CONFIG    = "fineweb-edu-dedup"
    # DEVIATION: ungated mirror; images have short side resized to 256 (see deviation table)
    IMAGENET_REPO  = "evanarlian/imagenet_1k_resized_256"
    CAULDRON_REPO  = "HuggingFaceM4/the_cauldron"
    MIX_RATIO      = {"text": 2, "imagenet": 2, "cauldron": 1}  # 144k:144k:72k, Table 1

    # ---- GQA evaluation (ADAPTATION: not in the paper) ----
    # Lowered from 1000 to protect the 12 h session: six training runs now cost up to
    # ~6 h, and fp32 greedy generation over the eval set runs twice (base + replacement).
    # Raise these if the training phase finishes early. Any reduction from 9,855 is
    # already a deviation; this makes it a larger one.
    EVAL_N          = 500
    GROUND_N        = 500
    MAX_NEW_TOKENS  = 16          # A10
    FULLFRAME_AREA  = 0.75        # A13
    IOU_THRESH      = 0.5         # A12
    MAP_THRESH_FRAC = 0.5         # A12

    # ---- rollout (Sec. 3.3, Eq. 8) ----
    ROLLOUT_K       = 4      # A7
    ROLLOUT_Q       = 0.25   # A7
    ROLLOUT_BLOCK   = 4      # A7

    # ---- attribution (Sec. 3.2, Sec. 4.2) ----
    RUN_ATTRIBUTION        = True
    ATTR_N_PROMPTS         = 3
    ATTR_TOP_FEATS_PER_LAY = 128
    ATTR_NODE_CUM          = 0.80   # Sec. 4.2
    ATTR_EDGE_CUM          = 0.98   # Sec. 4.2
    ATTR_LOGIT_NODES       = 10     # Sec. 4.2
    ATTR_LOGIT_CUM_PROB    = 0.95   # Sec. 4.2
    ATTR_EPS_FRAC          = 1e-4   # A9

    # ---- SMOKE TEST ----------------------------------------------------------
    # True  = exercise every cell end to end as fast as possible. The numbers it
    #         produces are NOT measurements and must not be reported.
    # False = the real reduced run described in the header.
    SMOKE_TEST = False

    SEED = 0

os.makedirs(CFG.OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Smoke-test overrides. Everything structural is left alone on purpose: all three
# TRANSCODER_LAYERS (so decoder truncation is exercised at 4, 16 and 28 layers, and
# layer 27 -- the configuration that has never trained -- is reached), and both
# TRAIN_MODES (so TRANSCODERS_MM is populated; with text-only it would be empty and
# CELL 6's replacement model would silently equal the base model while CELL 7's
# attribution graph degenerated to embedding-only edges -- wrong numbers, no crash).
# Only durations shrink.
# ---------------------------------------------------------------------------
if CFG.SMOKE_TEST:
    CFG.TRAIN_STEPS         = 3
    CFG.MAX_SECONDS_PER_RUN = 900
    CFG.BDEC_INIT_BATCHES   = 1
    CFG.FVU_EVAL_BATCHES    = 2
    CFG.EVAL_N              = 10
    CFG.GROUND_N            = 10
    CFG.ATTR_N_PROMPTS      = 2
    print("=" * 78)
    print("SMOKE TEST MODE — every number produced by this run is meaningless.")
    print("  purpose: prove all 10 cells execute end to end, including the untested")
    print("           layer-27 training and CELLS 6, 7 and 8.")
    print(f"  train steps {CFG.TRAIN_STEPS} (real run: 50), eval {CFG.EVAL_N} records "
          f"(real run: 500), grounding {CFG.GROUND_N}, attribution graphs {CFG.ATTR_N_PROMPTS}")
    print("  results are written to results_smoke.csv and every row is marked SMOKE TEST.")
    print("  set CFG.SMOKE_TEST = False for the real run.")
    print("=" * 78)

# ----------------------------- seeding --------------------------------------
random.seed(CFG.SEED); np.random.seed(CFG.SEED)
torch.manual_seed(CFG.SEED); torch.cuda.manual_seed_all(CFG.SEED)
print("Seeded python/numpy/torch with", CFG.SEED)
print("NONDETERMINISM THAT REMAINS: cuDNN/cuBLAS kernel selection and fp16 "
      "reduction order, TopK tie-breaking, HuggingFace streaming shard order, "
      "and GPU atomics in scatter. Greedy decoding is otherwise deterministic.")

# ----------------------------- hardware -------------------------------------
N_GPU = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()} | GPU count: {N_GPU}")
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name} {p.total_memory/2**30:.1f} GiB  sm_{p.major}{p.minor}")
    if CFG.DTYPE == "bfloat16" and p.major < 8:
        raise RuntimeError("bfloat16 requested but this GPU does not support it.")
DEVICE = torch.device("cuda:0" if N_GPU > 0 else "cpu")
TORCH_DTYPE = {"float16": torch.float16, "float32": torch.float32}[CFG.DTYPE]

import torch._dynamo                      # belt and braces alongside TORCHDYNAMO_DISABLE
torch._dynamo.config.disable = True
print(f"torch.compile disabled: TORCHDYNAMO_DISABLE={os.environ.get('TORCHDYNAMO_DISABLE')}, "
      f"torch._dynamo.config.disable={torch._dynamo.config.disable}")

# ----------------------------- mounts ---------------------------------------
print("\n--- MOUNT PATHS ---")
print("annotations :", CFG.ANN_PATH)
print("image root  :", CFG.IMAGE_ROOT)
if not os.path.exists(CFG.ANN_PATH):
    print("MOUNTED UNDER /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - 2
        if depth <= 3:
            print("  " * depth + os.path.basename(root) + f"/  ({len(files)} files)")
    raise FileNotFoundError(f"Annotation file not found: {CFG.ANN_PATH}")
if not os.path.isdir(CFG.IMAGE_ROOT):
    print("MOUNTED UNDER /kaggle/input:")
    for p in sorted(os.listdir("/kaggle/input")):
        print("  ", p)
    raise FileNotFoundError(f"IMAGE_ROOT is not a directory: {CFG.IMAGE_ROOT}")

image_files = os.listdir(CFG.IMAGE_ROOT)
print(f"files in IMAGE_ROOT: {len(image_files)}")

# ----------------------------- load annotations -----------------------------
records = []
with open(CFG.ANN_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print(f"records loaded: {len(records)} (expected {CFG.EXPECTED_RECORDS})")
if len(records) != CFG.EXPECTED_RECORDS:
    print(f"WARNING: record count differs from the {CFG.EXPECTED_RECORDS} stated.")

req = {"question","answer","full_answer","image","width","height","bboxs","dataset","split"}
missing_fields = req - set(records[0].keys())
if missing_fields:
    raise KeyError(f"Records are missing expected fields: {sorted(missing_fields)}")

unique_images = sorted({r["image"] for r in records})
print(f"unique images: {len(unique_images)} (expected {CFG.EXPECTED_UNIQUE_IMAGES})")
print(f"records per image: {len(records)/max(1,len(unique_images)):.3f}")

# ----------------------------- resolve every image --------------------------
present, absent = set(), []
for fn in unique_images:
    if os.path.exists(os.path.join(CFG.IMAGE_ROOT, fn)):
        present.add(fn)
    else:
        absent.append(fn)
print(f"resolved: {len(present)} / {len(unique_images)}   missing: {len(absent)}")
if absent:
    print("first 20 missing filenames:")
    for fn in absent[:20]:
        print("   ", fn)
    raise FileNotFoundError(
        f"{len(absent)} images did not resolve under IMAGE_ROOT. "
        "This is a path problem, not a missing dataset. No substitute images will be generated."
    )

# ----------------------------- size verification ----------------------------
# bboxs are pixel coordinates; a re-encoded/resized image puts the box in the wrong place.
real_size = {}
for fn in unique_images:
    with Image.open(os.path.join(CFG.IMAGE_ROOT, fn)) as im:
        real_size[fn] = im.size            # (w, h); header-only read, lazy

kept, dropped_mismatch = [], 0
for r in records:
    w_real, h_real = real_size[r["image"]]
    if int(r["width"]) != w_real or int(r["height"]) != h_real:
        dropped_mismatch += 1
        continue
    r["_w"], r["_h"] = w_real, h_real
    kept.append(r)
print(f"\ndropped for width/height mismatch: {dropped_mismatch} "
      f"({100*dropped_mismatch/max(1,len(records)):.2f}%)")
if dropped_mismatch > 0.05 * len(records):
    print("WARNING: a large fraction disagrees. The images were likely re-encoded "
          "and the pixel boxes cannot be trusted. Grounding numbers below are unreliable.")
if not kept:
    raise RuntimeError("Every record failed the size check. Boxes are unusable; stopping.")

# ----------------------------- clamp boxes ----------------------------------
clamped_boxes = 0
for r in kept:
    w, h = r["_w"], r["_h"]
    fixed = []
    for b in r["bboxs"]:
        x1, y1, x2, y2 = [float(v) for v in b]
        cx1, cy1 = min(max(x1, 0.0), w), min(max(y1, 0.0), h)
        cx2, cy2 = min(max(x2, 0.0), w), min(max(y2, 0.0), h)
        if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
            clamped_boxes += 1
        if cx2 > cx1 and cy2 > cy1:
            fixed.append([cx1, cy1, cx2, cy2])
    r["_boxes"] = fixed
print(f"boxes clamped to [0,width]x[0,height]: {clamped_boxes}")

kept = [r for r in kept if len(r["_boxes"]) > 0]
print(f"records with at least one valid box after clamping: {len(kept)}")

# ----------------------------- full-frame flag ------------------------------
def union_box(boxes):
    xs1 = min(b[0] for b in boxes); ys1 = min(b[1] for b in boxes)
    xs2 = max(b[2] for b in boxes); ys2 = max(b[3] for b in boxes)
    return [xs1, ys1, xs2, ys2]

n_full = 0
for r in kept:
    ub = union_box(r["_boxes"])
    r["_union"] = ub
    frac = ((ub[2]-ub[0]) * (ub[3]-ub[1])) / float(r["_w"] * r["_h"])
    r["_union_frac"] = frac
    r["_fullframe"] = bool(frac >= CFG.FULLFRAME_AREA)     # A13
    n_full += r["_fullframe"]
print(f"records flagged full-frame (union box >= {CFG.FULLFRAME_AREA} of image area): {n_full}")
print("Grounding accuracy will be reported twice: all records, and excluding full-frame records.")

# ----------------------------- answer distribution --------------------------
from collections import Counter

def host_ram_gib():
    """(resident, total) host RAM in GiB. The kernel death at ~2110 s produced no Python
    traceback -- the process was killed. Reading this each step turns that into an
    exception naming the actual limit."""
    rss = tot = float("nan")
    try:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    rss = int(line.split()[1]) / 1024 ** 2
                    break
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    tot = int(line.split()[1]) / 1024 ** 2
                    break
    except OSError as e:
        print(f"host RAM unreadable: {e}")
    return rss, tot

def check_host_ram(where):
    rss, tot = host_ram_gib()
    if math.isnan(rss) or math.isnan(tot):
        return rss, tot
    if rss > CFG.HOST_RAM_ABORT_FRAC * tot:
        raise MemoryError(
            f"Host RAM at {rss:.1f} / {tot:.1f} GiB "
            f"({100*rss/tot:.0f}%) during {where}. Aborting before the kernel is killed. "
            f"The usual cause is too many concurrent streaming iterators buffering decoded "
            f"images -- lower CFG.CAULDRON_TAKE_PER_VISIT / CFG.CAULDRON_SHUFFLE_BUFFER / "
            f"CFG.IMAGENET_SHUFFLE_BUFFER."
        )
    return rss, tot

_rss, _tot = host_ram_gib()
print(f"host RAM at start: {_rss:.1f} / {_tot:.1f} GiB")
ans_counts = Counter(str(r["answer"]).strip().lower() for r in kept)
print(f"\nval split: {len(kept)} records, {len(ans_counts)} unique answer strings")
print("10 most frequent answer strings and their counts:")
for a, c in ans_counts.most_common(10):
    print(f"   {a!r}: {c}")
print("There are no hard class labels and no probability scores in this task "
      "-> ROC-AUC and PR-AUC are N/A (see EVALUATE).")

VAL_RECORDS = kept
with open(f"{CFG.OUT_DIR}/gqa_val_clean.jsonl", "w") as f:
    for r in VAL_RECORDS:
        f.write(json.dumps({k: v for k, v in r.items() if not k.startswith("_")}) + "\n")
print(f"\nCELL 1 OK. clean records: {len(VAL_RECORDS)}")

In [ ]:
# ============================================================================
# CELL 1b — REPO REACHABILITY + ARCHITECTURE VERIFICATION
# Confirms every repo resolves without a token, then checks the loaded config
# against Sec. 4 "Architecture".
#
# Two tiers, deliberately:
#   HARD     — values this notebook actually sizes tensors from. A mismatch means
#              the transcoders would be built wrong, so it raises.
#   REPORTED — values the paper states but this notebook never consumes. Printed
#              and compared, never fatal. Sec. 4's "34 attention heads" is a known
#              erratum (see the Errata table in the header): Gemma-3-4B has 8 query
#              heads, 4 KV heads, head_dim 256. 2560/34 is not an integer, so the
#              paper's figure cannot describe any model; the layer count appears to
#              have been copied into the heads slot.
# ============================================================================
from huggingface_hub import model_info, dataset_info
from transformers import AutoConfig

print("--- repo reachability (no token) ---")
for kind, rid in (("model", CFG.MODEL_ID),
                  ("dataset", CFG.TEXT_REPO),
                  ("dataset", CFG.IMAGENET_REPO),
                  ("dataset", CFG.CAULDRON_REPO)):
    fn = model_info if kind == "model" else dataset_info
    info = fn(rid, token=None)          # token=None on purpose: proves it is ungated
    gated = getattr(info, "gated", False)
    print(f"  {kind:8s} {rid:45s} reachable, gated={gated}")
    if gated:
        raise RuntimeError(
            f"{rid} is gated (gated={gated}). Find an ungated mirror or set HF_TOKEN "
            f"with accepted terms, and add a deviation row for whichever you use."
        )

cfg = AutoConfig.from_pretrained(CFG.MODEL_ID, token=CFG.HF_TOKEN)
tcfg = cfg.text_config if hasattr(cfg, "text_config") else cfg
vcfg = cfg.vision_config if hasattr(cfg, "vision_config") else None

# ---- HARD: the notebook builds tensors from these three ----------------------
HARD = {
    "num_hidden_layers": (CFG.N_LAYERS, "Sec. 4: 34-layer transformer"),
    "hidden_size":       (CFG.D_MODEL,  "Sec. 4: d_model = 2560"),
    "intermediate_size": (CFG.D_FF,     "Sec. 4: MLP dimension d_ff = 10240"),
}
print("\n--- HARD checks (transcoder shapes depend on these) ---")
mismatched = []
for k, (want, cite) in HARD.items():
    got = getattr(tcfg, k)
    ok = (got == want)
    print(f"  {'OK ' if ok else 'MISMATCH'} {k}: config={got}  paper={want}   [{cite}]")
    if not ok:
        mismatched.append(k)
if mismatched:
    raise RuntimeError(
        f"Config disagrees with the paper on {mismatched}. These size the transcoders "
        f"(d_feat = N_latents * hidden_size, one per layer); building them against the "
        f"wrong shape would silently produce a different model. Do not proceed."
    )

# ---- REPORTED: recorded, never fatal ----------------------------------------
print("\n--- REPORTED config (not consumed by this notebook) ---")
PAPER_CLAIM = {"num_attention_heads": 34}    # Sec. 4 — known erratum
for k in ("num_attention_heads", "num_key_value_heads", "head_dim",
          "sliding_window", "rope_theta", "vocab_size"):
    got = getattr(tcfg, k, None)
    if k in PAPER_CLAIM and got != PAPER_CLAIM[k]:
        print(f"  ERRATUM {k}: config={got}  paper says={PAPER_CLAIM[k]} "
              f"-- paper figure is not reproducible ({CFG.D_MODEL}/{PAPER_CLAIM[k]} "
              f"is not an integer); using the config value. Not fatal: no tensor "
              f"in this notebook is sized from it.")
    else:
        print(f"  {k}: {got}")

hd = getattr(tcfg, "head_dim", None)
nh = getattr(tcfg, "num_attention_heads", None)
if hd and nh:
    print(f"  note: head_dim x num_heads = {hd*nh}, which is not hidden_size "
          f"({CFG.D_MODEL}) -- Gemma 3 decouples head_dim from hidden_size/num_heads.")

if vcfg is not None:
    print("\n--- vision tower (Sec. 4: SigLIP, patch 14, 896x896) ---")
    for k in ("patch_size", "image_size", "num_hidden_layers",
              "hidden_size", "num_attention_heads"):
        print(f"  {k}: {getattr(vcfg, k, None)}")
    n_vl = getattr(vcfg, "num_hidden_layers", None)
    if n_vl is not None and CFG.ROLLOUT_K > n_vl:
        raise RuntimeError(
            f"ROLLOUT_K={CFG.ROLLOUT_K} exceeds the vision tower depth ({n_vl}); "
            f"Eq. 8 cannot roll out over more layers than exist."
        )
    print(f"  rollout will use the last K={CFG.ROLLOUT_K} of {n_vl} vision layers (A7)")

print("\nArchitecture matches Sec. 4 on every value this notebook depends on. "
      "Treating the mirror as weight-equivalent to google/gemma-3-4b-it "
      "(unverified by hash — see deviation table).")
print("CELL 1b OK")

In [ ]:
# ============================================================================
# CELL 2 — LOAD DATA
# Transcoder training corpora as listed in Table 1 (streamed).
# Mixture ratio 144k : 144k : 72k = 2 : 2 : 1 preserved (Sec. 4.1 "Datasets").
# ============================================================================
from datasets import load_dataset, get_dataset_config_names
from itertools import cycle

def text_stream():
    ds = load_dataset(CFG.TEXT_REPO, CFG.TEXT_CONFIG, split="train",
                      streaming=True, token=CFG.HF_TOKEN)
    ds = ds.shuffle(seed=CFG.SEED, buffer_size=1000)
    for ex in ds:
        t = ex.get("text", None)
        if t:
            yield {"source": "text", "text": t, "image": None}

def imagenet_stream():
    # ASSUMPTION A14: paper does not specify the ImageNet caption text; using a template.
    # DEVIATION: ungated mirror with short side resized to 256 (see deviation table).
    ds = load_dataset(CFG.IMAGENET_REPO, split="train", streaming=True, token=CFG.HF_TOKEN)
    feats = ds.features
    if feats is None or "label" not in feats or not hasattr(feats["label"], "names"):
        raise RuntimeError(
            f"{CFG.IMAGENET_REPO} did not expose a ClassLabel 'label' feature; "
            f"cannot build captions. Got features={feats}."
        )
    names = feats["label"].names          # e.g. 'tench, Tinca tinca'
    if len(names) != 1000:
        raise RuntimeError(f"Expected 1000 ImageNet classes, got {len(names)}.")
    ds = ds.shuffle(seed=CFG.SEED, buffer_size=CFG.IMAGENET_SHUFFLE_BUFFER)
    for ex in ds:
        img, lbl = ex.get("image", None), ex.get("label", None)
        if img is None or lbl is None or lbl < 0:      # test split carries label = -1
            continue
        name = names[lbl].split(",")[0].strip()
        yield {"source": "imagenet",
               "text": f"This is a photo of a {name}.",
               "image": img.convert("RGB")}

# the_cauldron subset health. Some subsets (CLEVR is one) store Image features as
# absolute paths on the dataset author's filesystem, e.g.
#   /fsx/m4/datasets/downloads/extracted/.../CLEVR_v1.0/images/train/CLEVR_train_049349.png
# rather than embedded bytes, so decoding raises FileNotFoundError and they can never
# stream. That is a defect in the dataset, not in this notebook. Probe once, keep the
# subsets that work, and record every exclusion -- Sec. 4.1 samples evenly from 50
# subsets, so dropping any is a real deviation and goes in the results notes.
_CAULDRON = {"usable": None, "dropped": [], "skipped_examples": Counter()}

def _probe_cauldron_subsets(subsets):
    usable = []
    print(f"probing {len(subsets)} Cauldron subsets for streamable images ...")
    for sname in subsets:
        try:
            d = load_dataset(CFG.CAULDRON_REPO, sname, split="train",
                             streaming=True, token=CFG.HF_TOKEN)
            ex = next(iter(d))                    # datasets decodes the image here
            imgs = ex.get("images", [])
            if not imgs:
                raise ValueError("record has no 'images' field")
            imgs[0].convert("RGB")                # force a full decode
            usable.append(sname)
        except Exception as e:                    # noqa: BLE001 - classified, not swallowed
            _CAULDRON["dropped"].append((sname, type(e).__name__, str(e)[:140]))
    print(f"  usable: {len(usable)} / {len(subsets)}")
    for sname, etype, msg in _CAULDRON["dropped"]:
        print(f"  DROPPED {sname}: {etype}: {msg}")
    if not usable:
        raise RuntimeError(
            "No Cauldron subset yielded a decodable image. Every subset failed the probe; "
            "the multimodal split cannot be built. See the DROPPED lines above."
        )
    return usable

def cauldron_stream():
    """Round-robin over subsets with exactly ONE stream open at a time.

    Sec. 4.1: "we sampled from its 50 subsets evenly to ensure even coverage." Even
    coverage comes from the round-robin, not from holding every subset open. The earlier
    version kept 48 iterators alive with shuffle(buffer_size=100) each, buffering ~4,800
    decoded PIL images plus 48 parquet row groups, and the kernel was killed on host RAM.
    A 50-step run draws only ~125 Cauldron examples in total, so one pass of
    48 subsets x CAULDRON_TAKE_PER_VISIT is more than enough.

    Caveat, recorded rather than hidden: on a second pass a subset is reopened from the
    start, so its first records would repeat. A 50-step run does not reach a second pass.
    """
    subsets = get_dataset_config_names(CFG.CAULDRON_REPO, token=CFG.HF_TOKEN)
    print(f"Cauldron subsets found: {len(subsets)}")
    if _CAULDRON["usable"] is None:
        _CAULDRON["usable"] = _probe_cauldron_subsets(subsets)
    usable = list(_CAULDRON["usable"])
    dropped_runtime = set()
    visit = 0
    while True:
        progressed = False
        for sname in usable:
            if sname in dropped_runtime:
                continue
            d = load_dataset(CFG.CAULDRON_REPO, sname, split="train",
                             streaming=True, token=CFG.HF_TOKEN)
            if CFG.CAULDRON_SHUFFLE_BUFFER > 0:
                d = d.shuffle(seed=CFG.SEED + visit,
                              buffer_size=CFG.CAULDRON_SHUFFLE_BUFFER)
            it_s = iter(d)
            taken, fails = 0, 0
            while taken < CFG.CAULDRON_TAKE_PER_VISIT:
                try:
                    ex = next(it_s)
                except StopIteration:
                    break
                except (FileNotFoundError, OSError) as e:
                    fails += 1
                    _CAULDRON["skipped_examples"][sname] += 1
                    if fails >= CFG.CAULDRON_MAX_FAILS:
                        dropped_runtime.add(sname)
                        _CAULDRON["dropped"].append(
                            (sname, type(e).__name__,
                             f"{fails} decode failures mid-stream"))
                        print(f"  DROPPING {sname} mid-stream after {fails} "
                              f"decode failures: {type(e).__name__}")
                        break
                    continue
                imgs, texts = ex.get("images", []), ex.get("texts", [])
                if not imgs or not texts:
                    continue
                try:
                    img = imgs[0].convert("RGB")
                except (FileNotFoundError, OSError):
                    fails += 1
                    _CAULDRON["skipped_examples"][sname] += 1
                    continue
                qa = texts[0]
                txt = (qa.get("user", "") + " " + qa.get("assistant", "")).strip()
                taken += 1
                progressed = True
                yield {"source": "cauldron", "text": txt, "image": img}
            del it_s, d          # close before opening the next subset
        visit += 1
        if not progressed:
            raise RuntimeError(
                f"No Cauldron subset yielded a usable record in a full pass. "
                f"Skipped by subset: {dict(_CAULDRON['skipped_examples'])}"
            )

def mixed_stream(mode):
    """mode='text'       -> SmolLM2 text split only (Fig. 4 bottom, text-only arm)
       mode='multimodal' -> full text+image mixture (Fig. 4 bottom, multimodal arm)"""
    if mode == "text":
        yield from text_stream()
        return
    gens = {"text": text_stream(), "imagenet": imagenet_stream(), "cauldron": cauldron_stream()}
    plan = []
    for name, w in CFG.MIX_RATIO.items():
        plan += [name] * w
    for name in cycle(plan):
        yield next(gens[name])

# Leakage guard: the transcoder corpora contain no GQA data.
_VAL_IMAGE_SET = {r["image"] for r in VAL_RECORDS}
_TRAIN_SOURCES = ("text", "imagenet", "cauldron")
print("Leakage guard: training sources are", _TRAIN_SOURCES,
      "- disjoint from the GQA val image set by construction "
      f"({len(_VAL_IMAGE_SET)} GQA images, none referenced by the streams above). "
      "Every sample's 'source' tag is asserted against this tuple inside the train loop.")
print("CELL 2 OK")

In [ ]:
# ============================================================================
# CELL 3 — PREPROCESSING
# ============================================================================
import re, string
from transformers import AutoProcessor

# ASSUMPTION A17: the checkpoint ships a slow image processor and the transformers
# default for use_fast is version-dependent. Pinned explicitly to the variant the
# checkpoint was saved with, so preprocessing does not drift across versions.
processor = AutoProcessor.from_pretrained(CFG.MODEL_ID, token=CFG.HF_TOKEN,
                                          use_fast=False)
tokenizer = processor.tokenizer
print("processor loaded:", type(processor).__name__, "(use_fast=False, pinned)")

# ---------------------------------------------------------------------------
# ASSUMPTION A11: the paper specifies no answer normalisation.
#   lowercase
#   -> delete apostrophes with no replacement   ("men's"   -> "mens")
#   -> replace every other punctuation mark with a space ("hot-dog" -> "hot dog")
#   -> drop a leading article, but never the only token
#   -> collapse whitespace
# Hyphens become spaces rather than being deleted so that a hyphenated prediction
# and an unhyphenated gold answer compare equal; deleting them instead would make
# "hot-dog" normalise to "hotdog" and score as a miss against gold "hot dog".
# ---------------------------------------------------------------------------
_ARTICLES = {"a", "an", "the"}
_APOSTROPHES = "'\u2019\u02bc`"
_DROP_CHARS  = str.maketrans("", "", _APOSTROPHES)
_SPACE_CHARS = str.maketrans({c: " " for c in string.punctuation
                              if c not in _APOSTROPHES})

def normalize_answer(s: str) -> str:
    s = str(s).lower().strip()
    s = s.translate(_DROP_CHARS)
    s = s.translate(_SPACE_CHARS)
    toks = [t for t in s.split() if t]
    while len(toks) > 1 and toks[0] in _ARTICLES:
        toks = toks[1:]
    return " ".join(toks)

# Self-test. Prints every case so a failure names the input, not just "AssertionError".
_CASES = [
    ("The Hot-Dog.",   "hot dog"),
    ("  SOUR cream ",  "sour cream"),
    ("a man's hat",    "mans hat"),
    ("Yes",            "yes"),
    ("the",            "the"),      # lone article survives
    ("An apple",       "apple"),
    ("blue/green",     "blue green"),
    ("2",              "2"),
]
_fails = []
print("\nnormalize_answer self-test:")
for raw, want in _CASES:
    got = normalize_answer(raw)
    ok = (got == want)
    print(f"  {'OK ' if ok else 'FAIL'} {raw!r:18s} -> {got!r:14s} (want {want!r})")
    if not ok:
        _fails.append((raw, got, want))
if _fails:
    raise AssertionError(f"normalize_answer failed on {_fails}")

def build_vqa_messages(question: str):
    """ASSUMPTION A10: prompt format is not specified by the paper."""
    return [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{question}\nAnswer with a single word or short phrase."},
    ]}]

def encode_vqa(record):
    img = Image.open(os.path.join(CFG.IMAGE_ROOT, record["image"])).convert("RGB")
    text = processor.apply_chat_template(build_vqa_messages(record["question"]),
                                         add_generation_prompt=True, tokenize=False)
    # add_special_tokens=False: apply_chat_template(tokenize=False) already emits <bos>,
    # and the tokenizer would prepend a second one. Measured on this setup: the prompt
    # began '<bos><bos><start_of_turn>user', and Gemma answered with proper nouns
    # ('Richard', 'Park') instead of GQA answers, scoring 0/10 on base AND replacement.
    # Verified supported: add_special_tokens is a TextKwargs field that Gemma3Processor
    # forwards to the tokenizer.
    # images is a nested list: one inner list per text sample.
    return processor(text=[text], images=[[img]], return_tensors="pt",
                     add_special_tokens=False), img

def encode_training_batch(samples):
    """Batch for transcoder training. Text-only samples use TEXT_SEQ_LEN=2048 (Table 1);
    image samples are encoded with the processor at the model's native 896x896."""
    if all(s["image"] is None for s in samples):
        enc = tokenizer([s["text"] for s in samples], return_tensors="pt",
                        padding="max_length", truncation=True,
                        max_length=CFG.TEXT_SEQ_LEN)
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
    texts, images = [], []
    for s in samples:
        if s["image"] is None:
            continue                       # image batch: text-only members dropped
        msgs = [{"role": "user", "content": [{"type": "image"},
                                             {"type": "text", "text": s["text"][:1000]}]}]
        texts.append(processor.apply_chat_template(msgs, tokenize=False))
        images.append([s["image"]])        # nested: one inner list per text
    if not texts:
        raise RuntimeError("Multimodal batch contained no images after filtering.")
    # add_special_tokens=False for the same reason as encode_vqa: these texts are already
    # chat-templated. The text-only branch above feeds RAW text to the tokenizer and so
    # keeps the default, which correctly prepends a single <bos>.
    enc = processor(text=texts, images=images, return_tensors="pt",
                    padding=True, truncation=True, max_length=1024,
                    add_special_tokens=False)
    return dict(enc)

# Shape smoke test on one real record: proves the nested-image form is accepted
# and that the processor emits the 256 image tokens Sec. 4 describes.
_probe_enc, _probe_img = encode_vqa(VAL_RECORDS[0])
print(f"\nencode_vqa probe on {VAL_RECORDS[0]['image']}:")
for k, v in _probe_enc.items():
    if torch.is_tensor(v):
        print(f"  {k}: {tuple(v.shape)} {v.dtype}")
if "pixel_values" not in _probe_enc:
    raise RuntimeError("Processor returned no pixel_values; the image was not consumed.")
_img_tok = int((_probe_enc["input_ids"][0] == processor.image_token_id).sum()) \
           if hasattr(processor, "image_token_id") else None
print(f"  image tokens in the prompt: {_img_tok} (Sec. 4: 256 soft image tokens)")

# ---- prompt sanity ---------------------------------------------------------
# The smoke run scored 0/10 on BOTH the base and the replacement model while emitting
# proper nouns ('Richard', 'Park') instead of GQA-style answers. That points at the
# prompt, not the metric. The classic cause is a DOUBLED BOS: apply_chat_template with
# tokenize=False returns a string that already starts with <bos>, and the tokenizer then
# prepends another one. Gemma degrades badly on that -- it stops behaving like a chat
# model and continues text instead. Check it explicitly rather than assume.
_ids = _probe_enc["input_ids"][0].tolist()
_bos_id = tokenizer.bos_token_id
_n_bos = sum(1 for t in _ids if t == _bos_id)
_head = tokenizer.decode([t for t in _ids[:24]], skip_special_tokens=False)
_tail = tokenizer.decode([t for t in _ids[-40:]], skip_special_tokens=False)
print(f"  BOS token id {_bos_id} appears {_n_bos}x in the prompt")
print(f"  prompt head: {_head!r}")
print(f"  prompt tail: {_tail!r}")
if _n_bos != 1:
    raise RuntimeError(
        f"The prompt contains {_n_bos} BOS tokens; exactly 1 is required. "
        f"apply_chat_template(tokenize=False) emits <bos> itself, so the processor must be "
        f"called with add_special_tokens=False. Two BOS tokens make Gemma answer as a text "
        f"continuator rather than a chat model and every accuracy number meaningless; zero "
        f"BOS tokens is also wrong for Gemma."
    )
print(f"  prompt structure OK: exactly 1 BOS, {_img_tok} image tokens, "
      f"ends with the model turn marker")
del _probe_enc, _probe_img

print("\nCELL 3 OK")

In [ ]:
# ============================================================================
# CELL 4 — MODEL
#   4a Gemma-3-4B-it loader
#   4b TopK transcoder (Eq. 1, Eq. 2, Sec. 3.1)
#   4c FVU (Eq. 3) and error node (Eq. 4)
#   4d replacement-model hooks
#   4e SigLIP attention rollout (Eq. 8, Sec. 3.3)
#   4f steering / patching (Eq. 9, Eq. 10)
# ============================================================================
import torch.nn as nn, torch.nn.functional as F
from transformers import Gemma3ForConditionalGeneration

# ---------------------------- 4a ---------------------------------------------
def load_gemma(last_layer=None):
    """Loads Gemma sharded so that cuda:1 keeps room for the transcoder.

    Placement is not "auto across both cards evenly". An even split leaves ~6.5 GiB
    free per card, and the transcoder needs 6.25 GiB of parameters + gradients +
    AdamW moments plus activations, which does not fit. CFG.MAX_MEMORY pushes the
    model onto cuda:0 so cuda:1 keeps ~10.5 GiB free.
    """
    kw = dict(torch_dtype=TORCH_DTYPE, token=CFG.HF_TOKEN,
              attn_implementation=CFG.ATTN_TRAIN)
    need_gib = None
    if N_GPU >= 2:
        if CFG.MAX_MEMORY is not None:
            mm = {int(k): v for k, v in CFG.MAX_MEMORY.items()}
        else:
            mm = {}
            for i in range(N_GPU):
                tot = torch.cuda.get_device_properties(i).total_memory / 2 ** 30
                reserve = CFG.GPU0_RESERVE_GIB if i == 0 else CFG.GPU1_RESERVE_GIB
                mm[i] = f"{int((tot - reserve) * 1024)}MiB"
            print(f"computed max_memory from device totals: {mm}")
        kw["max_memory"] = mm
        # "sequential", NOT "auto". With device_map="auto" accelerate runs
        # get_balanced_memory, which rewrites max_memory to spread weights EVENLY and
        # overrides the caps below. Measured: caps of {0:11GiB,1:13GiB} and
        # {0:13GiB,1:13GiB} produced an identical 583/300 tensor split, i.e. raising the
        # cuda:0 cap moved nothing. "sequential" fills devices in order up to each cap,
        # which is what actually keeps weights off the transcoder's card.
        kw["device_map"] = "sequential"
        print(f"sharding sequentially with max_memory={kw['max_memory']}")
    elif N_GPU == 1 and CFG.DTYPE == "float32":
        need_gib = 4_300_079_472 * 4 / 2 ** 30
        raise RuntimeError(
            f"fp32 Gemma-3-4B is {need_gib:.2f} GiB and does not fit a single "
            f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB GPU, "
            f"let alone alongside the 6.25 GiB transcoder. This configuration needs 2 GPUs. "
            f"fp16 would fit but produces inf activations in the mid-stack (see CFG.DTYPE)."
        )
    m = Gemma3ForConditionalGeneration.from_pretrained(CFG.MODEL_ID, **kw)
    if N_GPU < 2:
        m = m.to(DEVICE)
    if last_layer is not None and CFG.TRUNCATE_DECODER:
        truncate_decoder(m, last_layer)
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def truncate_decoder(model, last_layer):
    """Drops decoder layers above `last_layer`.

    Exact, not an approximation: layer L's MLP input depends only on layers < L, so
    deleting everything above it cannot change the captured activation. Purely a cost
    measure -- the previous run spent ~63 s/step on every target layer because the full
    34-layer forward ran even when capturing layer 3.

    config.num_hidden_layers and config.layer_types are left alone: the mask builder
    reads layer_types to decide sliding vs full attention per layer, and each retained
    layer keeps its own layer_idx and attention_type.
    """
    layers = get_decoder_layers(model)
    n_before = len(layers)
    if last_layer >= n_before:
        raise IndexError(f"Target layer {last_layer} does not exist in a {n_before}-layer decoder.")

    before_alloc = [torch.cuda.memory_allocated(i) for i in range(N_GPU)]
    kept = nn.ModuleList([layers[i] for i in range(last_layer + 1)])
    removed = [layers[i] for i in range(last_layer + 1, n_before)]
    lm = model.model.language_model if hasattr(model.model, "language_model") else model.language_model
    lm.layers = kept

    # Dropping the modules from the ModuleList is NOT enough. accelerate's dispatch
    # hooks keep references to every submodule, so the removed layers stay alive and
    # their CUDA storages are never released. Measured: after truncating to 4 of 34
    # layers, live parameters were 5.47 GiB while the allocator still held 13.23 GiB,
    # and the next allocation OOMed. Detaching each storage frees the memory regardless
    # of who still holds the module object.
    try:
        from accelerate.hooks import remove_hook_from_module
        for m in removed:
            remove_hook_from_module(m, recurse=True)
    except ImportError:
        print("  accelerate.hooks unavailable; releasing storages only")
    for m in removed:
        for t in list(m.parameters(recurse=True)) + list(m.buffers(recurse=True)):
            t.data = torch.empty(0, device=t.data.device, dtype=t.data.dtype)
    del removed, layers
    torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()

    freed = [(before_alloc[i] - torch.cuda.memory_allocated(i)) / 2 ** 30 for i in range(N_GPU)]
    print(f"  decoder truncated: {n_before} -> {len(kept)} layers "
          f"(capturing layer {last_layer}; layers above it cannot affect it)")
    print(f"  released: " + ", ".join(f"cuda:{i} {freed[i]:.2f} GiB" for i in range(N_GPU)))
    return model

def get_decoder_layers(model):
    for path in (("model", "language_model", "layers"),
                 ("language_model", "model", "layers"),
                 ("model", "layers")):
        obj = model
        ok = True
        for a in path:
            if not hasattr(obj, a):
                ok = False; break
            obj = getattr(obj, a)
        if ok:
            return obj
    print(model)
    raise AttributeError("Could not locate the decoder layer list on this model object.")

def get_vision_layers(model):
    for path in (("model", "vision_tower", "vision_model", "encoder", "layers"),
                 ("vision_tower", "vision_model", "encoder", "layers")):
        obj = model; ok = True
        for a in path:
            if not hasattr(obj, a):
                ok = False; break
            obj = getattr(obj, a)
        if ok:
            return obj
    print(model)
    raise AttributeError("Could not locate the SigLIP encoder layers on this model object.")

def language_model_module(model):
    lm = model.model.language_model if hasattr(model.model, "language_model") else None
    if lm is None:
        lm = getattr(model, "language_model", None)
    if lm is None:
        raise AttributeError("Could not locate the language model submodule.")
    return lm

def set_attn_implementation(model, impl):
    """Sets the attention kernel on the DECODER only, never the vision tower.

    Two lessons are baked in here.

    (1) Scope. Setting eager everywhere made SigLIP materialise a
        1 x 16 x 4096 x 4096 fp32 = 1024 MiB attention tensor and OOM. The vision tower
        does not need freezing: Sec. 3.2 linearises the language model, CELL 7 treats the
        merged inputs_embeds as a graph LEAF, and the linearised pass calls the decoder
        directly without running the vision tower at all.

    (2) Reach. PreTrainedModel._from_config runs `config = copy.deepcopy(config)`, so
        every Gemma3Attention holds a deep copy of config.text_config. Assigning
        model.config.text_config._attn_implementation does not reach them; the layers
        must be walked. Measured: the top-level config said 'eager' while the decoder
        still ran sdpa and the softmax patch silently never fired.
    """
    lm = language_model_module(model)
    seen, n = set(), 0
    targets = [getattr(model.config, "text_config", None)]
    for m in lm.modules():
        targets.append(getattr(m, "config", None))
    for c in targets:
        if c is None or id(c) in seen:
            continue
        seen.add(id(c))
        c._attn_implementation = impl
        n += 1
    return n

def verify_attn_implementation(model, impl):
    """Reads the value back off a real attention module, not off model.config."""
    layers = get_decoder_layers(model)
    attn = layers[0].self_attn
    got = getattr(attn.config, "_attn_implementation", None)
    vis = None
    try:
        vlayers = get_vision_layers(model)
        vis = getattr(vlayers[0].self_attn.config, "_attn_implementation", None)
    except AttributeError:
        vis = "unavailable"
    print(f"  attention kernel: decoder layer 0 = '{got}', vision layer 0 = '{vis}'")
    if vis == "eager":
        raise RuntimeError(
            f"The vision tower is on eager attention. SigLIP runs {(896//14)**2} tokens x 16 "
            f"heads, which materialises a 1024 MiB attention tensor and will OOM. Only the "
            f"decoder should be switched."
        )
    if got != impl:
        raise RuntimeError(
            f"Decoder attention is '{got}', not '{impl}'. The config objects held by the "
            f"layers are deep copies of model.config.text_config, so they must be set "
            f"directly -- see set_attn_implementation."
        )
    return got

def disable_compiled_forwards(model):
    """torch.compile artifacts left over from CELL 6's generate() would execute a traced
    graph that never calls the Python F.softmax, defeating the linearisation. Clear them
    for the attribution pass and report if any were found."""
    n = 0
    for m in model.modules():
        if getattr(m, "_compiled_call_impl", None) is not None:
            m._compiled_call_impl = None
            n += 1
    if n:
        print(f"  cleared {n} compiled forward(s) so the linearised pass runs in eager Python")
    return n

# ---------------------------- 4b ---------------------------------------------
class TopKTranscoder(nn.Module):
    """Eq. 1: z(x)=ReLU(W_enc x + b_enc), sparsified by TopK (Sec. 3.1).
       Eq. 2: TC(x)=W_dec z(x) + b_dec."""
    def __init__(self, d_model, d_feat, k):
        super().__init__()
        W = torch.empty(d_feat, d_model)
        nn.init.kaiming_uniform_(W, a=math.sqrt(5))              # ASSUMPTION A3
        self.W_enc = nn.Parameter(W)
        self.b_enc = nn.Parameter(torch.zeros(d_feat))
        self.W_dec = nn.Parameter(W.t().contiguous().clone())    # ASSUMPTION A3 (tied init)
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        self.d_feat, self.d_model, self.k = d_feat, d_model, k

    def preact(self, x):                       # W_enc x + b_enc  (target side of Eq. 6)
        return F.linear(x, self.W_enc, self.b_enc)

    def encode(self, x):
        z = F.relu(self.preact(x))                         # Eq. 1
        vals, idx = torch.topk(z, self.k, dim=-1)          # TopK(z, k), Sec. 3.1
        z_sparse = torch.zeros_like(z).scatter_(-1, idx, vals)
        return z_sparse, idx, vals

    def decode(self, z):                       # Eq. 2
        return F.linear(z, self.W_dec, self.b_dec)

    def forward(self, x):
        z, idx, vals = self.encode(x)
        return self.decode(z), z, idx, vals

    @torch.no_grad()
    def calibrate_decoder_scale(self, x_sample, y_sample, alpha):
        """ASSUMPTION A3: rescale W_dec so the initial output perturbation is `alpha`
        times the target's own standard deviation.

        Without this the decoder's initial magnitude comes from the weight init and is
        the same regardless of what it is reconstructing, while the target's scale
        varies ~74x across layers (std 3.77 at layer 3, 0.051 at layer 15). Measured:
        layer 3 reached FVU 0.519 while layer 15 sat at 1.698 -- worse than the mean
        predictor -- because the init perturbation swamped a small-scale target.
        After this, the starting FVU is ~1 + alpha^2 at every layer.

        W_dec stays non-zero so W_enc still receives gradient; zeroing it would freeze
        the encoder.
        """
        z, _, _ = self.encode(x_sample)
        pert_std = (self.decode(z) - self.b_dec).std()
        tgt_std = (y_sample - self.b_dec).std()
        if not (torch.isfinite(pert_std) and pert_std > 0):
            raise RuntimeError(f"Decoder perturbation std is {float(pert_std)}; cannot calibrate.")
        if not (torch.isfinite(tgt_std) and tgt_std > 0):
            raise RuntimeError(f"Target std is {float(tgt_std)}; cannot calibrate.")
        factor = float(alpha * tgt_std / pert_std)
        self.W_dec.mul_(factor)
        return float(pert_std), float(tgt_std), factor

    @torch.no_grad()
    def init_decoder_bias(self, mean_y):
        """ASSUMPTION A3: seed b_dec with the mean of MLP(x) (Sparsify convention;
        Sec. 4.1 states the framework was built on top of Sparsify). With b_dec = 0 the
        transcoder cannot emit even the constant mean of the target, which puts the FVU
        floor at 1 + mean^2/Var before optimisation starts."""
        self.b_dec.copy_(mean_y.to(self.b_dec.device, self.b_dec.dtype))

def paper_lr(n_latents, d_model):
    """Sec. 4.1: lr = 2e-4 * sqrt(2^14 / (N_latents * d_model))."""
    return CFG.LR_BASE * math.sqrt(CFG.LR_REF_NUMER / (n_latents * d_model))

# ---------------------------- 4c ---------------------------------------------
def error_node(y, y_hat):
    """Eq. 4: e(x) = MLP(x) - TC(x), the reconstruction residual the paper tracks
    as a separate node in the circuit graph."""
    return y - y_hat

class StreamingFVU:
    """Eq. 3:  FVU = (1/n) sum_i (y_i - yhat_i)^2  /  (1/n) sum_i (y_i - ybar)^2
                   = MSE / Var(y).

    Accumulated in chunks rather than computed in one pass. A held-out batch is
    12 x 2048 = 24576 rows, and one 24576 x 81920 fp32 latent tensor is 7.50 GiB,
    so the whole batch cannot be encoded at once on a 14.56 GiB card.

    Chunking is exact here, not an approximation: carrying sum(y), sum(y^2) and the
    summed squared error lets Var be reconstructed about the true per-dimension mean
    over every row seen, which is what Eq. 3's denominator is. Averaging per-chunk
    FVUs instead would NOT give this -- each chunk's variance would be about its own
    local mean. Sums are kept in float64 so 10^7-row accumulation does not drift.
    """
    def __init__(self, d_model, device):
        self.d = d_model
        self.n = 0
        self.sum_y = torch.zeros(d_model, dtype=torch.float64, device=device)
        self.sum_y2 = torch.zeros(d_model, dtype=torch.float64, device=device)
        self.sse = torch.zeros((), dtype=torch.float64, device=device)
        self.sq_err_node = torch.zeros((), dtype=torch.float64, device=device)

    def update(self, y, y_hat):
        yd = y.double()
        e = error_node(yd, y_hat.double())          # Eq. 4
        self.sum_y += yd.sum(0)
        self.sum_y2 += (yd * yd).sum(0)
        self.sse += (e * e).sum()
        self.sq_err_node += (e * e).sum()
        self.n += y.shape[0]

    def result(self):
        """Returns (fvu, mse, var, err_node_rms). NaN where undefined, never a
        substitute value."""
        if self.n == 0:
            return float("nan"), float("nan"), float("nan"), float("nan")
        mse = float(self.sse) / (self.n * self.d)
        mean_y = self.sum_y / self.n
        var = float((self.sum_y2 / self.n - mean_y * mean_y).mean())
        rms = math.sqrt(float(self.sq_err_node) / (self.n * self.d))
        if not (var > 0):
            return float("nan"), mse, var, rms
        return mse / var, mse, var, rms

# ---------------------------- 4d ---------------------------------------------
class ActCapture:
    """Captures MLP input x and MLP output y = MLP(x) at chosen decoder layers."""
    def __init__(self, model, layer_ids):
        self.buf, self.handles = {}, []
        self.layer_ids = list(layer_ids)
        layers = get_decoder_layers(model)
        for li in layer_ids:
            mlp = layers[li].mlp
            self.handles.append(mlp.register_forward_hook(self._mk(li)))
    def _mk(self, li):
        def hook(module, args, output):
            self.buf[li] = (args[0].detach(), output.detach())
        return hook
    def clear(self): self.buf = {}
    def remove(self):
        for h in self.handles: h.remove()
        self.handles = []

class ReplacementModel:
    """Substitutes MLP outputs with TC(x) at the transcoded layers (Sec. 3.1).
    Layers without a trained transcoder keep their original MLP -> DEVIATION."""
    def __init__(self, model, transcoders):
        self.model, self.tcs, self.handles = model, transcoders, []
    def _mk(self, tc):
        def hook(module, args, output):
            x = args[0]
            dev = tc.W_enc.device
            z, _, _ = tc.encode(x.to(dev, torch.float32))
            return tc.decode(z).to(output.device, output.dtype)
        return hook
    def __enter__(self):
        layers = get_decoder_layers(self.model)
        for li, tc in self.tcs.items():
            self.handles.append(layers[li].mlp.register_forward_hook(self._mk(tc)))
        return self.model
    def __exit__(self, *a):
        for h in self.handles: h.remove()
        self.handles = []

# ---------------------------- 4e ---------------------------------------------
def freest_cuda_device(default):
    """Picks the CUDA device with the most free memory. The rollout needs working
    room and cuda:0 holds the bulk of the model."""
    if not torch.cuda.is_available():
        return default
    best, best_free = default, -1
    for i in range(torch.cuda.device_count()):
        free, _ = torch.cuda.mem_get_info(i)
        if free > best_free:
            best, best_free = torch.device(f"cuda:{i}"), free
    return best

@torch.no_grad()
def rollout_map(model, pixel_values):
    """Eq. 8 + Sec. 3.3, computed ONE HEAD AT A TIME.

    The SigLIP tower emits T = (896/14)^2 = 4096 patch tokens over 16 heads, so the
    all-head attention logits are 1 x 16 x 4096 x 4096 fp32 = 1024 MiB, and the softmax
    output doubles that to 2.00 GiB. cuda:0 carries the full model and had 0.70 GiB free,
    which is exactly the allocation that failed. Per head the same tensor is 64 MiB.

    Two passes over the heads: the first measures each head's mean row entropy, the
    second averages only the lowest-entropy fraction q (Sec. 3.3). q and k are kept
    (1 x 16 x 4096 x 72 fp32 = 18 MiB) so the second pass is a cheap recompute rather
    than 1 GiB of retained attention matrices.
    """
    vlayers = get_vision_layers(model)
    K = min(CFG.ROLLOUT_K, len(vlayers))
    sel = list(range(len(vlayers) - K, len(vlayers)))
    inputs, handles = {}, []
    def mk(i):
        def pre(mod, args, kwargs):
            h = args[0] if args else kwargs["hidden_states"]
            inputs[i] = h.detach()
        return pre
    for i in sel:
        handles.append(vlayers[i].register_forward_pre_hook(mk(i), with_kwargs=True))
    vt = model.model.vision_tower if hasattr(model.model, "vision_tower") else model.vision_tower
    vt(pixel_values=pixel_values.to(next(vt.parameters()).device,
                                    next(vt.parameters()).dtype))
    for h in handles:
        h.remove()

    work = freest_cuda_device(next(vt.parameters()).device)
    R = None
    for i in sel:
        blk = vlayers[i]
        h = inputs[i]
        h = blk.layer_norm1(h) if hasattr(blk, "layer_norm1") else h
        attn = blk.self_attn
        nh = attn.num_heads if hasattr(attn, "num_heads") else attn.config.num_attention_heads
        B, T, C = h.shape
        hd = C // nh
        q = attn.q_proj(h).view(B, T, nh, hd).transpose(1, 2).to(work, torch.float32)
        k = attn.k_proj(h).view(B, T, nh, hd).transpose(1, 2).to(work, torch.float32)
        scale = 1.0 / math.sqrt(hd)

        # pass 1: mean row entropy per head, one head at a time
        ent = torch.empty(nh, device=work, dtype=torch.float32)
        for j in range(nh):
            P = torch.softmax((q[0, j] @ k[0, j].transpose(-1, -2)) * scale, dim=-1)
            ent[j] = -(P * (P + 1e-12).log()).sum(-1).mean()
            del P
        n_keep = max(1, int(round(CFG.ROLLOUT_Q * nh)))          # ASSUMPTION A7
        keep = ent.topk(n_keep, largest=False).indices           # most focused heads

        # pass 2: average the selected heads, still one at a time
        A_bar = torch.zeros(T, T, device=work, dtype=torch.float32)
        for j in keep.tolist():
            A_bar += torch.softmax((q[0, j] @ k[0, j].transpose(-1, -2)) * scale, dim=-1)
        A_bar /= float(n_keep)
        del q, k

        A_t = A_bar + torch.eye(T, device=work)                  # identity residual
        del A_bar
        A_t /= A_t.sum(-1, keepdim=True)                         # row-stochastic
        R = A_t if R is None else (R @ A_t)
        if R is not A_t:
            del A_t
    inputs.clear()

    # ASSUMPTION A8: aggregate rows of R_vis into one saliency map
    sal = R.mean(0)
    del R
    g = int(math.isqrt(sal.numel()))
    if g * g != sal.numel():
        raise RuntimeError(f"Vision token count {sal.numel()} is not a square grid.")
    grid = sal.view(1, 1, g, g)
    b = CFG.ROLLOUT_BLOCK                                        # ASSUMPTION A7
    pooled = F.avg_pool2d(grid, kernel_size=b, stride=b)         # 64x64 -> 16x16 = 256
    pooled = pooled - pooled.min()
    pooled = pooled / (pooled.max() + 1e-12)                     # normalize each map
    return pooled[0, 0].float().cpu()

# ---------------------------- 4f ---------------------------------------------
class Steering:
    """Eq. 9: dz = v - z(x);  Eq. 10: h <- h + dz * d_{l,i}.

    Implemented by writing v into the latent before the decoder, which adds
    (v - z_i) * d_i to the MLP output and hence to the residual stream.

    Positions come from the attribution graph, which is built on a single PREFILL pass
    where the whole prompt is present, so z is (B, T=prompt_len, d_feat). During
    autoregressive decoding T collapses to 1 and that chunk holds a NEW absolute
    position, so a prefill index is both meaningless and out of range -- measured as
        IndexError: index 2 is out of bounds for dimension 1 with size 1
    A position-specific edit is therefore applied on the first forward through its layer
    only, which is the pass that computes that position. Targets with pos=None apply at
    every position on every step.
    """
    def __init__(self, model, transcoders, targets):
        # targets: list of (layer, position or None, feature_index, value)
        self.model, self.tcs, self.targets, self.handles = model, transcoders, targets, []
        self.n_forward, self.n_applied = {}, 0

    def _mk(self, li, tc):
        tg = [t for t in self.targets if t[0] == li]
        def hook(module, args, output):
            x = args[0]
            dev = tc.W_enc.device
            z, _, _ = tc.encode(x.to(dev, torch.float32))
            first = self.n_forward.get(li, 0) == 0
            for (_, pos, fi, val) in tg:
                if pos is None:
                    z[..., fi] = val
                    self.n_applied += 1
                elif first:
                    if pos >= z.shape[-2]:
                        raise IndexError(
                            f"Steering position {pos} is outside the prefill sequence "
                            f"(length {z.shape[-2]}) at layer {li}. Positions must come "
                            f"from an attribution graph built on the same prompt."
                        )
                    z[:, pos, fi] = val
                    self.n_applied += 1
            self.n_forward[li] = self.n_forward.get(li, 0) + 1
            return tc.decode(z).to(output.device, output.dtype)
        return hook

    def __enter__(self):
        self.n_forward, self.n_applied = {}, 0
        layers = get_decoder_layers(self.model)
        for li, tc in self.tcs.items():
            self.handles.append(layers[li].mlp.register_forward_hook(self._mk(li, tc)))
        return self.model

    def __exit__(self, *a):
        for h in self.handles: h.remove()
        self.handles = []

print("CELL 4 OK — transcoder, replacement hooks, rollout, steering defined")

In [ ]:
# ============================================================================
# CELL 5 — TRAIN
# One transcoder per layer in CFG.TRANSCODER_LAYERS, for each mode in
# CFG.TRAIN_MODES (text-only vs multimodal = the Fig. 4 bottom ablation).
# Loss = reconstruction error only (Sec. 3.1); sparsity comes from TopK alone.
# ============================================================================
import pandas as pd

D_FEAT = CFG.N_LATENTS * CFG.D_MODEL   # ASSUMPTION A1
LR = paper_lr(CFG.N_LATENTS, CFG.D_MODEL)
print(f"d_feat per layer = N_latents * d_model = {CFG.N_LATENTS} * {CFG.D_MODEL} = {D_FEAT}")
print(f"learning rate from Sec. 4.1 formula = {LR:.3e}")

GEMMA = None
GEMMA_DEVICE = None
GEMMA_PARAMS = None          # set from the FULL model reloaded after training

def audit_placement(model, tag):
    """Reports BYTES per device. A tensor count of 583/300 looked identical across two
    different max_memory settings and hid that placement had not changed."""
    hist = Counter(str(p.device) for p in model.parameters())
    nbytes = {}
    for p in model.parameters():
        d = str(p.device)
        nbytes[d] = nbytes.get(d, 0) + p.numel() * p.element_size()
    total = sum(nbytes.values())
    print(f"  placement [{tag}]: {total/2**30:.2f} GiB of live parameters")
    for d in sorted(nbytes):
        print(f"    {d}: {nbytes[d]/2**30:6.2f} GiB across {hist[d]} tensors")
    # Report what the ALLOCATOR holds too. These diverged badly once -- 5.47 GiB of live
    # parameters against 13.23 GiB allocated, because truncated layers were not released
    # -- and reporting only live parameters hid it until the next allocation OOMed.
    for i in range(N_GPU):
        alloc = torch.cuda.memory_allocated(i) / 2 ** 30
        free_b, tot_b = torch.cuda.mem_get_info(i)
        live = nbytes.get(f"cuda:{i}", 0) / 2 ** 30
        gap = alloc - live
        flag = "  <-- unreleased" if gap > 1.0 else ""
        print(f"    cuda:{i} allocator: {alloc:6.2f} GiB allocated, {free_b/2**30:5.2f} GiB free "
              f"(live params {live:.2f}, gap {gap:+.2f}){flag}")
    offloaded = [d for d in hist if d == "meta" or d.startswith("disk")]
    if offloaded:
        raise RuntimeError(
            f"Parameters on {offloaded} -- accelerate offloaded to disk because "
            f"CFG.MAX_MEMORY ({CFG.MAX_MEMORY}) could not hold the model. "
            f"A disk-offloaded run is not a usable measurement."
        )
    if N_GPU > 0 and any(d.startswith("cpu") for d in hist):
        raise RuntimeError(f"Parameters on CPU: {dict(hist)}. Adjust CFG.MAX_MEMORY.")
    return sum(p.numel() for p in model.parameters()), total

def acquire_model(last_layer):
    """Loads Gemma, truncated to layers 0..last_layer when last_layer is not None."""
    global GEMMA, GEMMA_DEVICE
    release_model()
    GEMMA = load_gemma(last_layer=last_layer)
    GEMMA_DEVICE = next(GEMMA.parameters()).device
    n_par, n_bytes = audit_placement(GEMMA, f"layers 0..{last_layer}" if last_layer is not None else "full")
    print(f"  inputs go to {GEMMA_DEVICE}")
    return n_par, n_bytes

def release_model():
    global GEMMA, GEMMA_DEVICE
    if GEMMA is not None:
        del GEMMA
        GEMMA, GEMMA_DEVICE = None, None
        torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()

# ---- static memory budget (independent of which layers are loaded) ----
_tc_params = 2 * D_FEAT * CFG.D_MODEL + D_FEAT + CFG.D_MODEL
_GiB = 2 ** 30
_tc_gib = _tc_params * 4 * 4 / _GiB   # params + grads + AdamW exp_avg + exp_avg_sq
print("\n--- memory budget ---")
print(f"  transcoder params+grad+AdamW (fp32): {_tc_gib:6.2f} GiB ({_tc_params} params x 4 copies)")
print(f"  capture forward micro-batched at {CFG.MICRO_BATCH} "
      f"(MLP transient {3*CFG.MICRO_BATCH*CFG.TEXT_SEQ_LEN*CFG.D_FF*4/_GiB:.2f} GiB "
      f"instead of {3*CFG.BATCH_SIZE*CFG.TEXT_SEQ_LEN*CFG.D_FF*4/_GiB:.2f} GiB)")
if CFG.TRUNCATE_DECODER:
    print(f"  decoder truncated per target layer, so the model is reloaded "
          f"{len(CFG.TRANSCODER_LAYERS)} times plus once in full for CELL 6/7")
if N_GPU == 0:
    print("  CPU only: this will run but the six transcoder runs will not finish in 12 h.")

def pick_device_for_transcoder():
    """With two cards the model owns cuda:0 and the transcoder gets cuda:1 outright.
    Activations cross the PCIe bus once per chunk, which is cheap next to giving the
    optimiser state a whole card."""
    if N_GPU == 0:
        return torch.device("cpu")
    if N_GPU >= 2:
        return torch.device("cuda:1")
    return torch.device("cuda:0")

def batch_iter(stream, bs):
    buf = []
    for s in stream:
        buf.append(s)
        if len(buf) == bs:
            yield buf; buf = []

def run_forward_capture(batch, cap, dest=None):
    """Forward pass whose only purpose is to populate the MLP hooks.

    Calls the base Gemma3Model, not the ForConditionalGeneration wrapper: the wrapper
    finishes with lm_head over every position (12 x 2048 x 262208 x 4B = 24 GiB at fp32)
    which nothing here reads. use_cache=False likewise skips a KV cache never read.

    The batch is encoded ONCE so padding is identical across micro-batches, then sliced
    along dim 0. Sample i's activations do not depend on sample j, so concatenating the
    micro-batch results reconstructs exactly the full-batch capture.
    """
    enc = encode_training_batch(batch)
    enc = {k: (v.to(GEMMA_DEVICE) if torch.is_tensor(v) else v) for k, v in enc.items()}
    if "pixel_values" in enc:
        enc["pixel_values"] = enc["pixel_values"].to(TORCH_DTYPE)

    n = enc["input_ids"].shape[0]
    mb = max(1, int(CFG.MICRO_BATCH))
    parts = {li: ([], []) for li in cap.layer_ids}

    for i in range(0, n, mb):
        sub = {}
        for k, v in enc.items():
            sub[k] = v[i:i + mb] if (torch.is_tensor(v) and v.shape[0] == n) else v
        cap.clear()
        with torch.no_grad():
            GEMMA.model(**sub, use_cache=False)
        if not cap.buf:
            raise RuntimeError("No MLP activations were captured; hook placement is wrong.")
        for li, (x, y) in cap.buf.items():
            tgt = dest if dest is not None else x.device
            parts[li][0].append(x.to(tgt, torch.float32))
            parts[li][1].append(y.to(tgt, torch.float32))
        cap.clear()
        del sub

    out = {li: (torch.cat(xs, 0), torch.cat(ys, 0)) for li, (xs, ys) in parts.items()}
    # NO torch.cuda.empty_cache() here. Calling it in this hot path under
    # PYTORCH_ALLOC_CONF=expandable_segments:True raised
    #   AcceleratorError: CUDA error: an illegal memory access was encountered
    # on the very next tensor op -- expandable segments unmap virtual address ranges
    # that in-flight work still references. expandable_segments already suppresses the
    # fragmentation empty_cache was meant to reclaim, so it is not needed here. Cache is
    # released only at phase boundaries, where nothing is in flight.
    got = out[cap.layer_ids[0]][0].shape[0]
    if got != n:
        raise RuntimeError(f"Micro-batching lost samples: reassembled {got} of {n}.")
    del enc, parts
    return out

def train_one(layer_id, mode):
    torch.manual_seed(CFG.SEED); np.random.seed(CFG.SEED); random.seed(CFG.SEED)
    dev = pick_device_for_transcoder()
    tc = TopKTranscoder(CFG.D_MODEL, D_FEAT, CFG.K_TOPK).to(dev, torch.float32)
    n_params = sum(p.numel() for p in tc.parameters())
    opt = torch.optim.AdamW(tc.parameters(), lr=LR)   # ASSUMPTION A4 for betas/eps/wd

    # ---- pre-flight memory check ----------------------------------------------
    # AdamW allocates exp_avg and exp_avg_sq lazily inside the FIRST opt.step(), so an
    # under-budgeted run gets through a whole forward and backward before dying. Check
    # now instead: 70 s of wasted work per run, times six runs, is worth one assertion.
    if dev.type == "cuda":
        _pbytes = n_params * 4
        _need = (
            _pbytes                                   # gradients
            + 2 * _pbytes                             # AdamW exp_avg + exp_avg_sq
            + 2 * CFG.BATCH_SIZE * CFG.TEXT_SEQ_LEN * CFG.D_MODEL * 4   # captured x, y
            + 3 * CFG.TOKEN_CHUNK * D_FEAT * 4        # chunk tensors live in backward
        )
        # Headroom for allocator caching and transient workspace. Kept deliberately
        # generous rather than tuned: the earlier 14.26 GiB failure turned out to be an
        # evenly-balanced model placement (~8 GiB of weights on cuda:1), not allocator
        # slop, and that is fixed by device_map="sequential". This remains a cheap early
        # guard against gross under-budgeting, not a proof that the run fits.
        _HEADROOM = 1.5 * 2**30
        _need_total = _need * 1.10 + _HEADROOM
        _free, _total = torch.cuda.mem_get_info(dev.index)
        print(f"  pre-flight on {dev}: {_free/2**30:.2f} GiB free | "
              f"{_need/2**30:.2f} GiB of tensors to allocate "
              f"(grads {_pbytes/2**30:.2f}, AdamW {2*_pbytes/2**30:.2f}, activations "
              f"{(_need-3*_pbytes)/2**30:.2f}) | "
              f"{_need_total/2**30:.2f} GiB required with headroom")
        if _free < _need_total:
            raise RuntimeError(
                f"Insufficient memory on {dev}: {_free/2**30:.2f} GiB free but "
                f"{_need_total/2**30:.2f} GiB required ({_need/2**30:.2f} GiB of tensors "
                f"plus {_HEADROOM/2**30:.1f} GiB measured allocator headroom). "
                f"Raise the cuda:0 cap in CFG.MAX_MEMORY to move model weights off this "
                f"card, or lower CFG.TOKEN_CHUNK. Do not lower CFG.BATCH_SIZE (paper fixes "
                f"it at 12, Sec. 4.1) or CFG.N_LATENTS below 32 (bottom of the Sec. 4.1 grid) "
                f"without adding a deviation row."
            )
    cap = ActCapture(GEMMA, [layer_id])
    stream = mixed_stream(mode)
    it = batch_iter(stream, CFG.BATCH_SIZE)

    warmup_steps = max(1, int(round(CFG.WARMUP_FRAC * CFG.TRAIN_STEPS)))
    def lr_at(step):     # linear warmup then constant (ASSUMPTION A5)
        return LR * min(1.0, (step + 1) / warmup_steps)

    # ---- decoder-bias initialisation (ASSUMPTION A3) --------------------------
    # Seed b_dec with the mean of MLP(x) over a few batches, the Sparsify convention
    # Sec. 4.1 says the framework was built on. With b_dec = 0 the transcoder cannot
    # emit even the constant mean of the target, so FVU starts at 1 + mean^2/Var and
    # must climb back to 1.0 before explaining any variance. The previous run measured
    # exactly that: error-node RMS equalled sqrt(MSE) to four digits on all three layers.
    _bd_sum = torch.zeros(CFG.D_MODEL, dtype=torch.float64, device=dev)
    _bd_rows = 0
    _cal_x = _cal_y = None
    with torch.no_grad():
        for _ in range(CFG.BDEC_INIT_BATCHES):
            b = next(it)
            bb = run_forward_capture(b, cap, dest=dev)
            x_init, y_init = bb[layer_id]
            x_init = x_init.reshape(-1, CFG.D_MODEL)
            y_init = y_init.reshape(-1, CFG.D_MODEL)
            _bd_sum += y_init.double().sum(0)
            _bd_rows += y_init.shape[0]
            if _cal_x is None:                     # keep one chunk for scale calibration
                _cal_x = x_init[:CFG.TOKEN_CHUNK].clone()
                _cal_y = y_init[:CFG.TOKEN_CHUNK].clone()
            del bb, x_init, y_init
    if _bd_rows == 0:
        raise RuntimeError("Decoder-bias init saw no rows; the stream yielded nothing.")
    check_host_ram(f"b_dec init, layer {layer_id} {mode}")
    _bd_mean = (_bd_sum / _bd_rows).float()
    tc.init_decoder_bias(_bd_mean)
    _implied_floor = 1.0   # FVU of the mean predictor, by definition
    print(f"  b_dec initialised from {_bd_rows} rows: |mean|={float(_bd_mean.abs().mean()):.4f} "
          f"max={float(_bd_mean.abs().max()):.4f} -> starting FVU floor is {_implied_floor:.1f}, "
          f"not 1 + mean^2/Var")
    if _cal_x is None:
        raise RuntimeError("No calibration sample was captured during b_dec init.")
    _p_std, _t_std, _factor = tc.calibrate_decoder_scale(_cal_x, _cal_y, CFG.DECODER_INIT_SCALE)
    print(f"  W_dec rescaled x{_factor:.4e}: init perturbation std {_p_std:.4e} -> "
          f"{CFG.DECODER_INIT_SCALE:.2f} x target std {_t_std:.4e}; "
          f"starting FVU ~{1 + CFG.DECODER_INIT_SCALE**2:.4f} regardless of layer scale")
    del _cal_x, _cal_y

    for d in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(d)
    t0 = time.time()
    step, seen_tokens = 0, 0
    tail_start = int((1 - CFG.DEAD_TAIL_FRAC) * CFG.TRAIN_STEPS)
    alive = torch.zeros(D_FEAT, dtype=torch.bool, device=dev)
    loss_log = []

    while step < CFG.TRAIN_STEPS and (time.time() - t0) < CFG.MAX_SECONDS_PER_RUN:
        batch = next(it)
        assert all(b["source"] in _TRAIN_SOURCES for b in batch), "unexpected training source"
        buf = run_forward_capture(batch, cap, dest=dev)
        x_all, y_all = buf[layer_id]
        x_all = x_all.reshape(-1, CFG.D_MODEL).to(dev, torch.float32)
        y_all = y_all.reshape(-1, CFG.D_MODEL).to(dev, torch.float32)

        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        opt.zero_grad(set_to_none=True)

        n_tok = x_all.shape[0]
        chunks = max(1, math.ceil(n_tok / CFG.TOKEN_CHUNK))
        step_loss = 0.0
        del buf
        for c in range(chunks):
            xs = x_all[c*CFG.TOKEN_CHUNK:(c+1)*CFG.TOKEN_CHUNK]
            ys = y_all[c*CFG.TOKEN_CHUNK:(c+1)*CFG.TOKEN_CHUNK]
            if xs.numel() == 0:
                continue
            y_hat, z, idx, vals = tc(xs)
            loss = F.mse_loss(y_hat, ys) * (xs.shape[0] / n_tok)   # ASSUMPTION A2
            loss.backward()
            step_loss += float(loss.detach())
            if step >= tail_start:
                alive[idx.reshape(-1)] = True
            del y_hat, z, vals
        if not math.isfinite(step_loss):
            xf = bool(torch.isfinite(x_all).all()); yf = bool(torch.isfinite(y_all).all())
            raise RuntimeError(
                f"Non-finite loss at step {step} (layer {layer_id}, mode {mode}, "
                f"dtype {CFG.DTYPE}). Captured MLP input finite={xf}, MLP output "
                f"finite={yf}. If either is False the model forward overflowed and the "
                f"target is unusable; if both are True the transcoder itself diverged."
            )
        opt.step()
        seen_tokens += n_tok
        loss_log.append({"step": step, "loss": step_loss, "lr": lr_at(step), "tokens": n_tok})
        step += 1
        _rss, _tot = check_host_ram(f"layer {layer_id} {mode} step {step}")
        if step % 25 == 0:
            print(f"  L{layer_id} {mode} step {step} loss {step_loss:.6f} "
                  f"elapsed {time.time()-t0:.0f}s tokens {seen_tokens} "
                  f"host RAM {_rss:.1f}/{_tot:.1f} GiB")

    train_time = time.time() - t0
    steps_done = step
    if steps_done == 0:
        raise RuntimeError("No training step completed; increase MAX_SECONDS_PER_RUN.")

    # ---- held-out FVU (Eq. 3), in-domain as in Sec. 4.1 ----
    # Chunked at TOKEN_CHUNK, same as training. A held-out batch is 24576 rows and
    # one 24576 x 81920 fp32 latent tensor is 7.50 GiB.
    tc.eval()
    acc = StreamingFVU(CFG.D_MODEL, dev)
    n_batches = 0
    with torch.no_grad():
        for _ in range(CFG.FVU_EVAL_BATCHES):
            batch = next(it)
            buf = run_forward_capture(batch, cap, dest=dev)
            x_all, y_all = buf[layer_id]
            x_all = x_all.reshape(-1, CFG.D_MODEL)
            y_all = y_all.reshape(-1, CFG.D_MODEL)
            del buf
            for c in range(0, x_all.shape[0], CFG.TOKEN_CHUNK):
                xs = x_all[c:c + CFG.TOKEN_CHUNK].to(dev, torch.float32)
                ys = y_all[c:c + CFG.TOKEN_CHUNK].to(dev, torch.float32)
                if xs.numel() == 0:
                    continue
                y_hat, z, idx, vals = tc(xs)
                acc.update(ys, y_hat)
                del y_hat, z, idx, vals
            n_batches += 1
    fvu_val, fvu_mse, fvu_var, err_rms = acc.result()
    if math.isnan(fvu_val):
        print(f"  FVU is NaN: n_rows={acc.n}, Var={fvu_var}. Not substituted.")
    else:
        print(f"  held-out FVU {fvu_val:.6f}  (MSE {fvu_mse:.6e} / Var {fvu_var:.6e}, "
              f"{acc.n} rows over {n_batches} batches); Eq.4 error-node RMS {err_rms:.6e}")

    # ---- dead latents (Fig. 4 top) ----
    tail_steps = steps_done - tail_start
    if tail_steps <= 0:
        dead_pct = float("nan")
        print(f"Dead PCT is NaN: run stopped at step {steps_done}, before the tail "
              f"window opened at {tail_start}. No dead-latent measurement was made.")
    else:
        dead_pct = 100.0 * float((~alive).sum().item()) / D_FEAT
        print(f"  Dead PCT {dead_pct:.2f}% measured over the last {tail_steps} step(s) "
              f"({tail_steps * CFG.BATCH_SIZE} sequences). Fig. 4 measures this over a "
              f"30,000-step run; a window this short cannot be compared to it -- with "
              f"k={CFG.K_TOPK} at most {tail_steps * CFG.BATCH_SIZE * CFG.TEXT_SEQ_LEN * CFG.K_TOPK} "
              f"latent slots can fire, against {D_FEAT} latents.")

    peak = max([torch.cuda.max_memory_allocated(d) for d in range(N_GPU)]) / 2**30 if N_GPU else float("nan")

    # Release gradients and optimiser state before returning. zero_grad runs at the START
    # of each step, so after the final step .grad is still populated -- measured as an
    # allocator gap of exactly 2.01x and 4.01x the parameter size on cuda:1 when one and
    # two trained transcoders were being kept. Only the parameters are needed downstream.
    tc.zero_grad(set_to_none=True)
    del opt
    torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()

    cap.remove()
    _ckpt = f"{CFG.OUT_DIR}/transcoder_L{layer_id}_{mode}.pt"
    torch.save({"state_dict": {k: v.cpu() for k, v in tc.state_dict().items()},
                "layer": layer_id, "mode": mode, "d_feat": D_FEAT, "k": CFG.K_TOPK},
               _ckpt)
    if not os.path.exists(_ckpt):
        raise RuntimeError(f"Checkpoint {_ckpt} was not written.")
    if CFG.SMOKE_TEST:
        # Exercise the write path, then reclaim the space: six checkpoints is ~9.6 GiB.
        _sz = os.path.getsize(_ckpt) / 2 ** 30
        os.remove(_ckpt)
        print(f"  checkpoint write verified ({_sz:.2f} GiB) and removed (smoke test)")
    pd.DataFrame(loss_log).to_csv(f"{CFG.OUT_DIR}/losslog_L{layer_id}_{mode}.csv", index=False)

    return {"tc": tc, "layer": layer_id, "mode": mode, "fvu": fvu_val,
            "dead_pct": dead_pct, "train_time": train_time, "steps": steps_done,
            "params": n_params, "peak_mem": peak, "tokens": seen_tokens,
            "fvu_rows": acc.n, "err_node_rms": err_rms, "tail_steps": max(0, tail_steps),
            "bdec_rows": _bd_rows, "dec_scale": _factor, "target_std": _t_std}

# ---- activation finiteness probe -------------------------------------------
# Runs ONE forward per target layer before any training starts. The previous run
# spent 27 minutes finishing layer 3 only to hit inf on layer 15's first batch;
# this surfaces the same problem in ~45 s, and reports which layer and how large.
def probe_layer_finiteness(layer_ids):
    """One forward before training. Reports max|x|, max|y| and non-finite counts so a
    precision problem surfaces in seconds rather than after a full training run."""
    print(f"  finiteness probe ({CFG.DTYPE}) ...")
    cap = ActCapture(GEMMA, layer_ids)
    probe_stream = mixed_stream("text")
    batch = [next(probe_stream) for _ in range(CFG.BATCH_SIZE)]
    buf = run_forward_capture(batch, cap)
    bad = []
    for li in layer_ids:
        x, y = buf[li]
        # nan_to_num, not boolean-mask indexing. x[torch.isfinite(x)] calls nonzero(),
        # which materialises 62.9M x 3 dims x int64 = 1.41 GiB of indices for a 0.23 GiB
        # tensor, and that allocation is what OOMed here.
        xf, yf = bool(torch.isfinite(x).all()), bool(torch.isfinite(y).all())
        xm = float(torch.nan_to_num(x.abs(), nan=0.0, posinf=0.0, neginf=0.0).max())
        ym = float(torch.nan_to_num(y.abs(), nan=0.0, posinf=0.0, neginf=0.0).max())
        n_bad = int((~torch.isfinite(y)).sum())
        print(f"    layer {li:2d}: MLP in finite={xf} max|x|={xm:.1f} | "
              f"MLP out finite={yf} max|y|={ym:.1f} non-finite elems={n_bad}")
        if not (xf and yf):
            bad.append(li)
    cap.remove()
    del buf
    torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()
    if bad:
        raise RuntimeError(
            f"Non-finite MLP activations at layer(s) {bad} in {CFG.DTYPE} before any "
            f"training. The target MLP(x) is itself inf, so no loss is definable. "
            f"fp16 max is 65504 and Gemma-3's residual stream exceeds it mid-stack."
        )

TRAIN_RESULTS, TRANSCODERS_MM = [], {}

# Layer-outer, mode-inner: the model is truncated to layers 0..L, so both modes for a
# given layer share one load.
for li in CFG.TRANSCODER_LAYERS:
    print(f"\n{'='*70}\nloading model truncated to layers 0..{li}\n{'='*70}")
    acquire_model(li)
    probe_layer_finiteness([li])
    for mode in CFG.TRAIN_MODES:
        print(f"\n=== training transcoder: layer {li}, mode {mode} ===")
        r = train_one(li, mode)
        TRAIN_RESULTS.append({k: v for k, v in r.items() if k != "tc"})
        if mode == "multimodal":
            # Park on CPU. CELL 6/7 need these, but nothing during training does, and
            # each one otherwise occupies 1.56 GiB of the card the next transcoder wants.
            TRANSCODERS_MM[li] = r["tc"].to("cpu")
            print(f"  transcoder L{li} parked on CPU ({sum(p.numel()*p.element_size() for p in TRANSCODERS_MM[li].parameters())/2**30:.2f} GiB)")
        del r["tc"]
        torch.cuda.synchronize(); gc.collect(); torch.cuda.empty_cache()

# CELL 6 and 7 need every layer and the lm_head, so reload the model in full.
print(f"\n{'='*70}\nreloading full 34-layer model for evaluation\n{'='*70}")
GEMMA_PARAMS, _ = acquire_model(None)
print(f"Gemma-3-4B-it parameters (measured, full model): {GEMMA_PARAMS}")

# Restore the parked transcoders to the GPU for CELL 6/7.
_tc_dev = pick_device_for_transcoder()
for _li in sorted(TRANSCODERS_MM):
    TRANSCODERS_MM[_li] = TRANSCODERS_MM[_li].to(_tc_dev)
if TRANSCODERS_MM:
    _tc_bytes = sum(p.numel() * p.element_size()
                    for m in TRANSCODERS_MM.values() for p in m.parameters())
    print(f"restored {len(TRANSCODERS_MM)} transcoder(s) to {_tc_dev}: {_tc_bytes/2**30:.2f} GiB")
    if _tc_dev.type == "cuda":
        _f, _t = torch.cuda.mem_get_info(_tc_dev.index)
        print(f"  {_tc_dev} now has {_f/2**30:.2f} GiB free")
        if _f < 1 * 2 ** 30:
            raise RuntimeError(
                f"Only {_f/2**30:.2f} GiB free on {_tc_dev} after restoring transcoders; "
                f"CELL 6 generation and CELL 7 attribution need working room."
            )

if _CAULDRON["dropped"]:
    print(f"\nCauldron subsets excluded: {len(_CAULDRON['dropped'])} "
          f"of {len(_CAULDRON['usable']) + len(_CAULDRON['dropped'])}")
    for sname, etype, msg in _CAULDRON["dropped"]:
        print(f"  {sname}: {etype}: {msg}")
    print(f"  records skipped mid-stream: {dict(_CAULDRON['skipped_examples'])}")
    print("  Sec. 4.1 samples evenly from 50 subsets; this run did not. Recorded in results.")

print("\nCELL 5 OK — transcoders trained:",
      [(r["layer"], r["mode"], r["steps"]) for r in TRAIN_RESULTS])

In [ ]:
# ============================================================================
# CELL 6 — EVALUATE (A): GQA normalized exact match
# ADAPTATION. The paper contains no VQA evaluation; this measures whether the
# transcoder-replaced model (Sec. 3.1) preserves the base model's behaviour.
# ============================================================================
rng = random.Random(CFG.SEED)
eval_pool = list(VAL_RECORDS)
rng.shuffle(eval_pool)
EVAL_RECORDS = eval_pool[:min(CFG.EVAL_N, len(eval_pool))]
print(f"eval subset: {len(EVAL_RECORDS)} of {len(VAL_RECORDS)} clean val records "
      f"(of {CFG.EXPECTED_RECORDS} in the file) — see deviation table")

_eval_ids = {id(r) for r in EVAL_RECORDS}
assert len(_eval_ids) == len(EVAL_RECORDS), "duplicate records in the eval subset"
print("Train/eval disjointness: transcoders were trained on SmolLM2/ImageNet/Cauldron "
      "only; no GQA image or question entered training. Overlap = 0 by construction.")

ans_eval = Counter(normalize_answer(r["answer"]) for r in EVAL_RECORDS)
print(f"eval subset: {len(ans_eval)} unique normalized answers; "
      f"most frequent covers {100*ans_eval.most_common(1)[0][1]/len(EVAL_RECORDS):.2f}% of records")

@torch.no_grad()
def generate_answer(model, record):
    enc, _ = encode_vqa(record)
    enc = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in enc.items()}
    if "pixel_values" in enc:
        enc["pixel_values"] = enc["pixel_values"].to(TORCH_DTYPE)
    n_in = enc["input_ids"].shape[1]
    out = model.generate(**enc, max_new_tokens=CFG.MAX_NEW_TOKENS,
                         do_sample=False, num_beams=1)     # greedy, ASSUMPTION A10
    return tokenizer.decode(out[0, n_in:], skip_special_tokens=True).strip()

def score_exact_match(model, records, tag):
    for d in range(N_GPU):
        torch.cuda.reset_peak_memory_stats(d)
    t0 = time.time()
    n_correct, preds = 0, []
    for i, r in enumerate(records):
        pred = generate_answer(model, r)
        gold = normalize_answer(r["answer"])
        hit = int(normalize_answer(pred) == gold)
        n_correct += hit
        preds.append({"image": r["image"], "question": r["question"],
                      "gold": r["answer"], "pred": pred, "hit": hit})
        if (i + 1) % 100 == 0:
            print(f"  [{tag}] {i+1}/{len(records)} elapsed {time.time()-t0:.0f}s")
    acc = n_correct / len(records)
    # Print real examples. A bare 0.0 cannot distinguish "the model answered badly" from
    # "the model was never asked the question properly".
    print(f"  [{tag}] first {min(5, len(preds))} predictions:")
    for _p in preds[:5]:
        print(f"    q={_p['question'][:60]!r} gold={_p['gold']!r} pred={_p['pred']!r} "
              f"hit={_p['hit']}")
    _empty = sum(1 for _p in preds if not normalize_answer(_p["pred"]))
    _long = sum(1 for _p in preds if len(normalize_answer(_p["pred"]).split()) > 4)
    print(f"  [{tag}] {_empty}/{len(preds)} predictions normalise to empty, "
          f"{_long}/{len(preds)} are longer than 4 words")
    peak = max([torch.cuda.max_memory_allocated(d) for d in range(N_GPU)]) / 2**30 if N_GPU else float("nan")
    with open(f"{CFG.OUT_DIR}/preds_{tag}.jsonl", "w") as f:
        for p in preds:
            f.write(json.dumps(p) + "\n")
    return {"acc": acc, "n": len(records), "time": time.time() - t0, "peak_mem": peak}

print("\n--- base Gemma-3-4B-it ---")
BASE_EM = score_exact_match(GEMMA, EVAL_RECORDS, "base")

print("\n--- transcoder-replaced model (partial: layers "
      f"{sorted(TRANSCODERS_MM.keys())} of {CFG.N_LAYERS}) ---")
with ReplacementModel(GEMMA, TRANSCODERS_MM) as rep_model:
    REP_EM = score_exact_match(rep_model, EVAL_RECORDS, "replacement")

for name, res in (("base", BASE_EM), ("replacement", REP_EM)):
    if res["acc"] in (0.0, 1.0, 0.5):
        print(f"WARNING: {name} accuracy is exactly {res['acc']}. This came from a real "
              f"comparison of {res['n']} generated strings against ground truth, but a clean "
              f"0.0/0.5/1.0 usually indicates a broken prompt or decoding path — inspect "
              f"{CFG.OUT_DIR}/preds_{name}.jsonl before trusting it.")
print("CELL 6 OK")

In [ ]:
# ============================================================================
# CELL 7 — EVALUATE (B): rollout grounding + attribution graph
# ============================================================================
# ---------------------------- B1: grounding ---------------------------------
ground_pool = [r for r in EVAL_RECORDS][:min(CFG.GROUND_N, len(EVAL_RECORDS))]
print(f"grounding subset: {len(ground_pool)} records "
      f"({sum(r['_fullframe'] for r in ground_pool)} flagged full-frame)")

def map_to_box(m, W, H):
    """ASSUMPTION A12: bbox of all pixels >= MAP_THRESH_FRAC * max of the map."""
    up = F.interpolate(m[None, None], size=(H, W), mode="bilinear", align_corners=False)[0, 0]
    thr = CFG.MAP_THRESH_FRAC * float(up.max())
    ys, xs = torch.where(up >= thr)
    if xs.numel() == 0:
        return None
    return [float(xs.min()), float(ys.min()), float(xs.max()) + 1, float(ys.max()) + 1]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

for d in range(N_GPU):
    torch.cuda.reset_peak_memory_stats(d)
if N_GPU:
    for d in range(N_GPU):
        _f, _t = torch.cuda.mem_get_info(d)
        print(f"  cuda:{d} free before rollout: {_f/2**30:.2f} GiB")
    print("  rollout runs on whichever card has the most free memory; peak is ~0.3 GiB "
          "with per-head attention (all-head would be 2.0 GiB)")
t0 = time.time()
hits_all, hits_nofull, n_all, n_nofull, ious = 0, 0, 0, 0, []
pred_fracs, gt_fracs = [], []      # box area as a fraction of the image
for i, r in enumerate(ground_pool):
    enc, img = encode_vqa(r)
    if "pixel_values" not in enc:
        raise RuntimeError("Processor returned no pixel_values; cannot compute rollout.")
    m = rollout_map(GEMMA, enc["pixel_values"])
    pb = map_to_box(m, r["_w"], r["_h"])
    if pb is None:
        raise RuntimeError(f"Rollout map for {r['image']} was empty above threshold.")
    v = iou(pb, r["_union"])
    ious.append(v)
    _img_area = float(r["_w"] * r["_h"])
    pred_fracs.append((pb[2] - pb[0]) * (pb[3] - pb[1]) / _img_area)
    gt_fracs.append(r["_union_frac"])
    hit = int(v >= CFG.IOU_THRESH)
    hits_all += hit; n_all += 1
    if not r["_fullframe"]:
        hits_nofull += hit; n_nofull += 1
    _every = 1 if len(ground_pool) <= 20 else 100
    if (i + 1) % _every == 0:
        _el = time.time() - t0
        print(f"  [grounding] {i+1}/{len(ground_pool)} elapsed {_el:.0f}s "
              f"({_el/(i+1):.1f}s/image, projected {_el/(i+1)*len(ground_pool):.0f}s total)")

GROUND_ALL = hits_all / n_all if n_all else float("nan")
GROUND_NOFULL = hits_nofull / n_nofull if n_nofull else float("nan")
GROUND_TIME = time.time() - t0
GROUND_PEAK = max([torch.cuda.max_memory_allocated(d) for d in range(N_GPU)]) / 2**30 if N_GPU else float("nan")
if n_nofull == 0:
    print("Grounding excluding full-frame boxes is N/A: every record in the subset was full-frame.")
for nm, v in (("all", GROUND_ALL), ("excl. full-frame", GROUND_NOFULL)):
    if v in (0.0, 1.0, 0.5):
        print(f"WARNING: grounding accuracy ({nm}) is exactly {v}; verify the rollout maps "
              "and the box threshold before treating it as a measurement.")
# ---- diagnostics -----------------------------------------------------------
# A hit rate of 0.0 is ambiguous on its own: the rollout map could be wrong, or it
# could simply be diffuse, which makes the thresholded box cover most of the frame and
# drives IoU down against a small ground-truth box. The paper flags this failure mode
# itself (Sec. 7: the attention maps "sometimes fail to localize relevant regions").
# Report the distributions so the two cases are distinguishable. Thresholds are NOT
# tuned to improve the score.
_iou = np.array(ious); _pf = np.array(pred_fracs); _gf = np.array(gt_fracs)
print(f"\n  grounding diagnostics over {len(_iou)} records:")
print(f"    IoU            mean {_iou.mean():.4f}  median {np.median(_iou):.4f}  "
      f"max {_iou.max():.4f}  (hit threshold {CFG.IOU_THRESH})")
print(f"    predicted box  mean area {_pf.mean():.3f} of image, median {np.median(_pf):.3f}, "
      f"{int((_pf > 0.9).sum())}/{len(_pf)} cover >90% of the frame")
print(f"    ground-truth   mean area {_gf.mean():.3f} of image, median {np.median(_gf):.3f}")
_ceil = float(np.mean(np.minimum(_gf, _pf) / np.maximum(_gf, _pf)))
print(f"    area-ratio ceiling on IoU given these box sizes: {_ceil:.4f} "
      f"(IoU cannot exceed the ratio of the smaller box to the larger)")
_cell = 1.0 / (256.0)          # one cell of the 16x16 pooled map (Sec. 4: 256 tokens)
if _pf.mean() > 0.9:
    print("    -> predicted boxes are essentially the WHOLE FRAME: the rollout map is "
          "diffuse, not mislocalised.")
elif _pf.mean() < 3 * _cell:
    print(f"    -> predicted boxes average {_pf.mean()/_cell:.1f} cells of the 16x16 pooled "
          f"map (one cell = {_cell:.4f} of the image). The map is EXTREMELY PEAKED: only a "
          f"handful of cells clear {CFG.MAP_THRESH_FRAC} x max, so the box is far smaller "
          f"than the ground-truth region and IoU >= {CFG.IOU_THRESH} is unreachable by "
          f"construction (ceiling above).")
    print("       This is a property of the map, not a scoring bug. ViT attention rollout "
          "is commonly dominated by a few high-norm 'sink' tokens, and Sec. 7 reports the "
          "same failure mode: the maps 'sometimes fail to localize relevant regions'.")
    print("       Levers, all ADAPTATIONS rather than paper values: ASSUMPTION A8 (mean "
          "over rows of R_vis), CFG.MAP_THRESH_FRAC, CFG.ROLLOUT_K, CFG.ROLLOUT_Q. "
          "None has been tuned to raise the score.")
np.savez(f"{CFG.OUT_DIR}/grounding_diagnostics.npz",
         iou=_iou, pred_area_frac=_pf, gt_area_frac=_gf)

# ---------------------------- B2: attribution graph -------------------------
# Eqs. 5-7 via stop-gradient linearisation (Sec. 3.2). Nonlinearities frozen:
# attention softmax detached, RMSNorm scale detached, TopK mask detached,
# non-transcoded MLP outputs detached (ASSUMPTION A16 / deviation).
import contextlib
from transformers.models.gemma3 import modeling_gemma3 as MG

_SOFTMAX_CALLS = {"n": 0}

@contextlib.contextmanager
def linearized():
    """Freezes nonlinearities per Sec. 3.2. Requires eager attention: sdpa and flash
    fuse the softmax into a kernel that never calls F.softmax, so the patch below would
    be a no-op and the graph would not be linearised. _SOFTMAX_CALLS counts invocations
    so a silent no-op is caught rather than producing a wrong attribution graph."""
    orig_softmax = F.softmax
    orig_norm_fwd = MG.Gemma3RMSNorm.forward
    _SOFTMAX_CALLS["n"] = 0
    def det_softmax(x, dim=None, dtype=None, **kw):
        _SOFTMAX_CALLS["n"] += 1
        return orig_softmax(x, dim=dim, dtype=dtype, **kw).detach()
    def det_norm(self, x):
        scale = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).detach()
        return (x.float() * scale * (1.0 + self.weight.float())).type_as(x)
    F.softmax = det_softmax
    torch.nn.functional.softmax = det_softmax
    MG.Gemma3RMSNorm.forward = det_norm
    try:
        yield
    finally:
        F.softmax = orig_softmax
        torch.nn.functional.softmax = orig_softmax
        MG.Gemma3RMSNorm.forward = orig_norm_fwd

def attribution_graph(record):
    layers = get_decoder_layers(GEMMA)
    tc_layers = sorted(TRANSCODERS_MM.keys())
    enc, _ = encode_vqa(record)
    enc = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in enc.items()}
    if "pixel_values" in enc:
        enc["pixel_values"] = enc["pixel_values"].to(TORCH_DTYPE)

    # pass 1: clean activations, to freeze TopK masks and get z values
    cap = ActCapture(GEMMA, tc_layers)
    embeds_holder = {}
    lm = GEMMA.model.language_model if hasattr(GEMMA.model, "language_model") else GEMMA.language_model
    def grab(mod, args, kwargs):
        if "inputs_embeds" in kwargs and kwargs["inputs_embeds"] is not None:
            embeds_holder["e"] = kwargs["inputs_embeds"].detach()
    h = lm.register_forward_pre_hook(grab, with_kwargs=True)
    with torch.no_grad():
        GEMMA(**enc)
    h.remove(); cap.remove()
    if "e" not in embeds_holder:
        raise RuntimeError("Could not capture merged inputs_embeds for the linearised pass.")

    clean = {}
    for li in tc_layers:
        x, _ = cap.buf[li]
        tc = TRANSCODERS_MM[li]
        z, idx, vals = tc.encode(x.reshape(-1, CFG.D_MODEL).to(tc.W_enc.device, torch.float32))
        clean[li] = {"z": z, "shape": x.shape}

    # pass 2: linearised forward with z as leaves
    z_leaves, handles = {}, []
    def mk(li):
        tc = TRANSCODERS_MM[li]
        def hook(module, args, output):
            zl = clean[li]["z"].clone().requires_grad_(True)
            z_leaves[li] = zl
            out = tc.decode(zl).view(clean[li]["shape"])
            return out.to(output.device, output.dtype)
        return hook
    for li in tc_layers:
        handles.append(layers[li].mlp.register_forward_hook(mk(li)))
    for li in range(CFG.N_LAYERS):
        if li not in tc_layers:
            handles.append(layers[li].mlp.register_forward_hook(
                lambda m, a, o: o.detach()))          # ASSUMPTION A16

    emb_leaf = embeds_holder["e"].clone().float().requires_grad_(True)
    with linearized():
        out = lm(inputs_embeds=emb_leaf.to(TORCH_DTYPE),
                 attention_mask=enc.get("attention_mask", None))
        hs = out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]
        # Slice to the final position BEFORE lm_head. Only the last position's logits
        # are attributed, and running lm_head over the whole prompt would build a
        # (seq x 262208) tensor plus its autograd graph for nothing.
        logits = GEMMA.lm_head(hs[:, -1:, :])[0, -1].float()
    for hd in handles: hd.remove()
    if _SOFTMAX_CALLS["n"] == 0:
        raise RuntimeError(
            "The F.softmax patch was never invoked during the linearised forward, so "
            f"attention was not frozen (Sec. 3.2). attn_implementation is "
            f"'{GEMMA.config._attn_implementation}' -- it must be 'eager'."
        )

    probs = torch.softmax(logits, -1)
    order = probs.argsort(descending=True)
    cum, logit_nodes = 0.0, []
    for t in order[:CFG.ATTR_LOGIT_NODES]:            # Sec. 4.2: <=10 logit nodes
        logit_nodes.append(int(t))
        cum += float(probs[t].detach())      # probs carries grad; detach before scalarising
        if cum >= CFG.ATTR_LOGIT_CUM_PROB:            # Sec. 4.2: cumulative mass >= 0.95
            break

    edges = []
    def backprop(scalar, target_name):
        grads = torch.autograd.grad(scalar, list(z_leaves.values()) + [emb_leaf],
                                    retain_graph=True, allow_unused=True)
        for li, g in zip(z_leaves.keys(), grads[:-1]):
            if g is None:
                continue
            a = z_leaves[li].detach()
            A = a * g                                  # Eq. 5: A = a_s * w_{s->t}
            nz = A.abs().reshape(-1)
            topn = min(CFG.ATTR_TOP_FEATS_PER_LAY, nz.numel())
            sel = nz.topk(topn).indices
            for s in sel.tolist():
                pos, fi = divmod(s, A.shape[-1])
                edges.append({"src": f"feat_L{li}_p{pos}_f{fi}", "dst": target_name,
                              "A": float(A.reshape(-1)[s])})
        ge = grads[-1]
        if ge is not None:
            per_tok = (ge * emb_leaf.detach()).sum(-1)[0]   # embedding node contribution
            for p in per_tok.abs().topk(min(32, per_tok.numel())).indices.tolist():
                edges.append({"src": f"emb_p{p}", "dst": target_name,
                              "A": float(per_tok[p])})

    for t in logit_nodes:
        backprop(logits[t], f"logit_{t}")
    for li in tc_layers[1:]:
        tc = TRANSCODERS_MM[li]
        z = clean[li]["z"]
        top = z.abs().reshape(-1).topk(min(CFG.ATTR_TOP_FEATS_PER_LAY, z.numel())).indices
        for s in top.tolist()[:8]:                    # cap targets: T4 budget (deviation)
            pos, fi = divmod(s, z.shape[-1])
            pre = z_leaves[li].reshape(-1)[s]
            if pre.requires_grad:
                backprop(pre, f"feat_L{li}_p{pos}_f{fi}")

    if not edges:
        raise RuntimeError("Attribution produced no edges; the linearised graph is disconnected.")
    mx = max(abs(e["A"]) for e in edges)
    eps = CFG.ATTR_EPS_FRAC * mx                      # ASSUMPTION A9
    edges = [e for e in edges if abs(e["A"]) >= eps]  # Sec. 3.2: prune |A| < eps
    edges.sort(key=lambda e: -abs(e["A"]))
    tot = sum(abs(e["A"]) for e in edges)
    keep, run = [], 0.0
    for e in edges:                                   # Sec. 4.2: edge threshold 0.98
        keep.append(e); run += abs(e["A"])
        if run / tot >= CFG.ATTR_EDGE_CUM:
            break
    infl = Counter()
    for e in keep:
        infl[e["src"]] += abs(e["A"])
    tot_n = sum(infl.values()); run, nodes = 0.0, []
    for n, v in infl.most_common():                   # Sec. 4.2: node threshold 0.80
        nodes.append(n); run += v
        if run / tot_n >= CFG.ATTR_NODE_CUM:
            break
    keep = [e for e in keep if e["src"] in set(nodes)]
    return {"image": record["image"], "question": record["question"],
            "nodes": nodes, "edges": keep,
            "logit_nodes": [tokenizer.decode([t]) for t in logit_nodes]}

ATTR_GRAPHS = []
if CFG.RUN_ATTRIBUTION:
    # Attribution needs eager attention (see linearized()). Safe here where the prompt
    # is ~289 tokens; it would not fit at the 2048-token training sequence length.
    _prev_attn = GEMMA.config._attn_implementation
    _n_cfg = set_attn_implementation(GEMMA, CFG.ATTN_ATTRIBUTION)
    print(f"attention implementation: {_prev_attn} -> {CFG.ATTN_ATTRIBUTION} "
          f"for attribution ({_n_cfg} config objects updated)")
    verify_attn_implementation(GEMMA, CFG.ATTN_ATTRIBUTION)
    disable_compiled_forwards(GEMMA)
    for r in EVAL_RECORDS[:CFG.ATTR_N_PROMPTS]:
        print(f"attribution graph for {r['image']} ...")
        ATTR_GRAPHS.append(attribution_graph(r))
    with open(f"{CFG.OUT_DIR}/attribution_graphs.json", "w") as f:
        json.dump(ATTR_GRAPHS, f, indent=2)
    for g in ATTR_GRAPHS:
        print(f"  {g['image']}: {len(g['nodes'])} nodes, {len(g['edges'])} edges kept")

    # ---- intervention: zero-ablate the top-influence feature (Eqs. 9-10) ----
    inter_log = []
    for g, r in zip(ATTR_GRAPHS, EVAL_RECORDS[:CFG.ATTR_N_PROMPTS]):
        feat_nodes = [n for n in g["nodes"] if n.startswith("feat_L")]
        if not feat_nodes:
            print(f"  no feature node survived pruning for {g['image']}; skipping ablation")
            continue
        li, pos, fi = feat_nodes[0][len("feat_L"):].split("_")
        li, pos, fi = int(li), int(pos[1:]), int(fi[1:])
        with ReplacementModel(GEMMA, TRANSCODERS_MM) as rep:
            before = generate_answer(rep, r)
        _steer = Steering(GEMMA, TRANSCODERS_MM, [(li, pos, fi, 0.0)])
        with _steer as st:
            after = generate_answer(st, r)
        if _steer.n_applied == 0:
            raise RuntimeError(
                f"Steering hook fired but applied no edit for {feat_nodes[0]}. The target "
                f"position never appeared in a forward pass, so the reported 'after' "
                f"output would be identical to 'before' for the wrong reason."
            )
        inter_log.append({"image": g["image"], "feature": feat_nodes[0],
                          "before": before, "after": after,
                          "edits_applied": _steer.n_applied,
                          "changed": bool(before != after)})
        print(f"  ablated {feat_nodes[0]} ({_steer.n_applied} edit(s) applied): "
              f"{before!r} -> {after!r}"
              + ("" if before != after else "   [output unchanged]"))
    with open(f"{CFG.OUT_DIR}/interventions.json", "w") as f:
        json.dump(inter_log, f, indent=2)
    set_attn_implementation(GEMMA, _prev_attn)
    print(f"attention implementation restored to {_prev_attn}")
print("CELL 7 OK")

In [ ]:
# ============================================================================
# CELL 8 — SAVE RESULTS
# Every cell traces to a variable computed in TRAIN or EVALUATE.
# No numeric literals here other than the paper's own thresholds.
# ============================================================================
import pandas as pd

SMOKE = bool(CFG.SMOKE_TEST)
FIDELITY_PREFIX = "SMOKE TEST (not a measurement) — " if SMOKE else ""
RESULTS_PATH = f"{CFG.OUT_DIR}/results_smoke.csv" if SMOKE else f"{CFG.OUT_DIR}/results.csv"
if SMOKE:
    print("=" * 78)
    print("SMOKE TEST — the table below proves the cells execute. The numbers are not")
    print(f"results: {CFG.TRAIN_STEPS} training steps and {CFG.EVAL_N} eval records.")
    print("=" * 78)

INPUTS = f"{CFG.ANN_PATH} | {CFG.IMAGE_ROOT}"
SOURCE = "Yang et al., Circuit Tracing in Vision-Language Models (CVPR Findings 2026)"
MODEL_STR = f"{CFG.MODEL_ID} ({GEMMA_PARAMS} params)"

def na(x):
    return "N/A" if (isinstance(x, float) and math.isnan(x)) else x

rows = []

# --- paper's own metrics first: FVU (Eq. 3) and Dead PCT (Fig. 4) ---
for r in TRAIN_RESULTS:
    rows.append({
        "Acc": "N/A", "Prec": "N/A", "Recall": "N/A", "F1": "N/A",
        "ROC-AUC": "N/A", "PR-AUC": "N/A", "FPR": "N/A", "FNR": "N/A",
        "Train Time": r["train_time"], "Params": r["params"],
        "Comm Cost": "N/A", "Training Steps": r["steps"],
        "n_eval": r["tokens"], "source": SOURCE, "split": f"transcoder-corpus/{r['mode']}",
        "model": f"transcoder L{r['layer']} N_latents={CFG.N_LATENTS} k={CFG.K_TOPK} d_feat={D_FEAT}",
        "inputs": (f"HF: {CFG.TEXT_REPO} | {CFG.IMAGENET_REPO} | {CFG.CAULDRON_REPO}"
                   + (f" (cauldron: {len(_CAULDRON['usable'])} of "
                      f"{len(_CAULDRON['usable']) + len(_CAULDRON['dropped'])} subsets usable)"
                      if r["mode"] == "multimodal" else "")),
        "fidelity": FIDELITY_PREFIX + "reduced",
        "FVU": na(r["fvu"]), "Dead_PCT": na(r["dead_pct"]),
        "Peak_GPU_Mem_GB": na(r["peak_mem"]), "Eval_Time_s": 0.0,
        "ErrNode_RMS": na(r["err_node_rms"]), "FVU_rows": r["fvu_rows"],
        "notes": (f"b_dec seeded from {r['bdec_rows']} rows and W_dec rescaled "
                  f"x{r['dec_scale']:.2e} to {CFG.DECODER_INIT_SCALE} x target std "
                  f"{r['target_std']:.3e} (A3); "
                  f"Eq.3 FVU over {r['fvu_rows']} held-out rows; Eq.4 error-node RMS; "
                  f"Fig.4 Dead PCT over the last {r['tail_steps']} step(s) only -- not "
                  f"comparable to Fig.4's 30,000-step curves. Paper ran "
                  f"{CFG.PAPER_STEPS} steps, this ran {r['steps']}"),
    })

# --- adaptation: GQA normalized exact match ---
for tag, res, mdl in (("base", BASE_EM, MODEL_STR),
                      ("replacement", REP_EM,
                       f"{MODEL_STR} + transcoders L{sorted(TRANSCODERS_MM.keys())}")):
    rows.append({
        "Acc": res["acc"], "Prec": "N/A", "Recall": "N/A", "F1": "N/A",
        "ROC-AUC": "N/A", "PR-AUC": "N/A", "FPR": "N/A", "FNR": "N/A",
        "Train Time": 0.0, "Params": GEMMA_PARAMS,
        "Comm Cost": "N/A", "Training Steps": 0,
        "n_eval": res["n"], "source": SOURCE, "split": "gqa_cot_val",
        "model": mdl, "inputs": INPUTS, "fidelity": FIDELITY_PREFIX + "adaptation",
        "FVU": "N/A", "Dead_PCT": "N/A",
        "Peak_GPU_Mem_GB": na(res["peak_mem"]), "Eval_Time_s": res["time"],
        "ErrNode_RMS": "N/A", "FVU_rows": "N/A",
        "notes": "normalized exact match; paper defines no VQA metric",
    })

# --- adaptation: rollout grounding (Eq. 8) ---
for tag, acc, n in (("grounding_all", GROUND_ALL, n_all),
                    ("grounding_excl_fullframe", GROUND_NOFULL, n_nofull)):
    rows.append({
        "Acc": na(acc), "Prec": "N/A", "Recall": "N/A", "F1": "N/A",
        "ROC-AUC": "N/A", "PR-AUC": "N/A", "FPR": "N/A", "FNR": "N/A",
        "Train Time": 0.0, "Params": GEMMA_PARAMS,
        "Comm Cost": "N/A", "Training Steps": 0,
        "n_eval": n, "source": SOURCE, "split": f"gqa_cot_val/{tag}",
        "model": f"SigLIP attention rollout (K={CFG.ROLLOUT_K}, q={CFG.ROLLOUT_Q}, b={CFG.ROLLOUT_BLOCK})",
        "inputs": INPUTS, "fidelity": FIDELITY_PREFIX + "adaptation",
        "FVU": "N/A", "Dead_PCT": "N/A",
        "Peak_GPU_Mem_GB": na(GROUND_PEAK), "Eval_Time_s": GROUND_TIME,
        "ErrNode_RMS": "N/A", "FVU_rows": "N/A",
        "notes": f"IoU>={CFG.IOU_THRESH} vs union of bboxs; paper defines no grounding metric",
    })

COLS = ["Acc","Prec","Recall","F1","ROC-AUC","PR-AUC","FPR","FNR",
        "Train Time","Params","Comm Cost","Training Steps",
        "n_eval","source","split","model","inputs","fidelity",
        "FVU","Dead_PCT","ErrNode_RMS","FVU_rows","Peak_GPU_Mem_GB","Eval_Time_s","notes"]
results = pd.DataFrame(rows)[COLS]

for c in ("Acc","FVU","Dead_PCT"):
    for i, v in results[c].items():
        if isinstance(v, float) and v in (0.0, 0.5, 1.0):
            print(f"WARNING row {i} column {c} = {v}. Computed from real data "
                  f"({results.loc[i,'n_eval']} items) but a clean 0.0/0.5/1.0 warrants inspection.")

results.to_csv(RESULTS_PATH, index=False)
pd.set_option("display.width", 250, "display.max_columns", 50)
display(results)
print(f"\nwritten: {RESULTS_PATH}")
print("ROC-AUC and PR-AUC are N/A: the task emits short answer strings, not class "
      "probabilities. No score independent of the label exists, so any AUC built from "
      "correctness flags would return 1.0 by construction and measure nothing.")
print("Prec/Recall/F1/FPR/FNR are N/A: open-vocabulary string matching has no positive "
      "class and no confusion matrix; there is no averaging scheme to report, and the "
      "paper specifies none.")
print("Comm Cost is N/A: the paper measures no communication cost.")

## Run notes

- **Do not pin numpy, pandas, pillow, or torch.** Cell 0 records their container versions and passes them to pip as a constraints file. Replacing a compiled package inside a live kernel leaves 1.x Python files sitting on 2.x `.so` files, and the next import dies with `ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject` — the dtype struct is 88 bytes in numpy 1.x and 96 in 2.x. Only a kernel restart clears it, which papermill cannot do mid-notebook. Cell 0 asserts none of them moved and runs a numpy/pandas smoke test before handing off. If you add a dependency, add it to `PINS`, never to the constraints file.
- **No token needed.** `unsloth/gemma-3-4b-it` and `evanarlian/imagenet_1k_resized_256` are both ungated, as are `HuggingFaceTB/smollm-corpus` and `HuggingFaceM4/the_cauldron`. Cell 1b proves this with `token=None` before any GPU time is spent. If any of them is gated later, Cell 1b raises rather than falling through to a 401 mid-training.
- **This configuration requires 2 GPUs.** fp32 Gemma-3-4B is 16.02 GiB and does not fit one 14.56 GiB T4. fp16 would fit but is not usable: measured here, layer-3 activations are finite in fp16 while layer-15 MLP output overflows and the captured target contains `inf` before any gradient step. Cell 4 raises on 1 GPU rather than loading something that cannot work.
- **Placement is audited, not assumed.** accelerate *warns* about disk offload and continues; a model with weights on the meta device either crashes deep in a forward or returns silently wrong numbers. Cell 5 prints a parameter device histogram and raises on any `meta`, `disk`, or `cpu` placement.
- **Micro-batching is memory-only.** `MICRO_BATCH = 3` splits the capture forward into 4 passes of 3. The batch is encoded once so padding is identical, and samples do not interact, so the concatenated result reconstructs the full-batch capture. All 24,576 tokens still form one optimiser step at the paper's batch size of 12. The Gemma MLP holds three `12x2048x10240` fp32 intermediates at once (2.81 GiB) inside `down_proj(act(gate(x)) * up(x))`; at micro-batch 3 that drops to 0.70 GiB.
- **Wall clock.** Six transcoder runs (3 layers × 2 modes) at `MAX_SECONDS_PER_RUN = 2100` cap out at ~3.5 h, leaving room for evaluation inside a 12 h session. Lower `EVAL_N`/`GROUND_N` if you need headroom — but any reduction from 9,855 belongs in the deviation table, and a subset score is not the paper's score.
- **Multi-GPU.** `load_gemma()` uses `device_map="auto"` only when two GPUs are visible; transcoders go to whichever GPU has the most free memory. The notebook runs unchanged on 1 GPU or CPU.
- **Outputs written to `/kaggle/working/`:** `results.csv`, `gqa_val_clean.jsonl`, `transcoder_L{layer}_{mode}.pt`, `losslog_L{layer}_{mode}.csv`, `preds_base.jsonl`, `preds_replacement.jsonl`, `grounding_ious.npy`, `attribution_graphs.json`, `interventions.json`.

- **Host RAM, not just GPU.** Kaggle gives ~30 GiB of host RAM and kills the kernel without a Python traceback when it runs out. Cell 5 reads `/proc/self/status` every step and raises a `MemoryError` naming the limit at 88% occupancy. The usual cause is concurrent streaming iterators buffering decoded images — `CAULDRON_TAKE_PER_VISIT`, `CAULDRON_SHUFFLE_BUFFER`, `IMAGENET_SHUFFLE_BUFFER` are the dials.

### What is *not* reproduced

Sections 3.4, 4.4, and 5 rest on human expert annotation — "we use human experts to discover and annotate circuits" (Sec. 3.4). The notebook produces the attribution graph those experts would read; grouping features into named circuit nodes, and the case studies (Mars/space shuttle, six-finger, visual arithmetic), are not automatable from the paper's description and are not attempted here.